# 0) Install & Login & Drive Mount

In [1]:
!pip install -q wandb polars geonamescache pycountry # xgboost imblearn networkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.0/32.0 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 181.1 MB/s eta 0:00:00


In [2]:
import torch, sys, os
print("torch:", torch.__version__)

torch: 2.10.0+cu128


In [3]:
!pip install -q torch-scatter torch-sparse torch-cluster torch-spline-conv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 7.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done


In [4]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.7 MB/s eta 0:00:00


In [6]:
import wandb
wandb.login()
# 계정 만들었으면, 2 누르고 본인 api 키 입력

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cho08040326 (0326byeol-korea-ac-kr) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1) Imports

In [8]:
import torch
print(torch.__version__)
print(torch.version.cuda)

2.10.0+cu128
12.8


In [9]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,  # AUPRC/PR curve 핵심
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

# from xgboost import XGBClassifier
# from imblearn.over_sampling import SMOTE
from collections import Counter
import random

import joblib
import pickle
from sklearn import set_config

import re
import pycountry
import geonamescache

# import networkit as nk
from collections import deque
from datetime import timedelta

import torch
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter, Linear
from typing import Union, Tuple, Optional

from torch_geometric.typing import (Adj, Size, OptTensor, PairTensor)
from torch_sparse import SparseTensor, set_diag
from torch_geometric.data import Data
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import remove_self_loops, add_self_loops, softmax
from torch_geometric.nn.inits import glorot, zeros
from torch_geometric.loader import NeighborLoader

#  import torch_xla.core.xla_model as xm

In [10]:
def seed_everything(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 2) 테이블 병합(Polars + Parquet)



In [ ]:
# # 파일 경로 설정 (본인의 경로에 맞게 수정하세요)
# base_path = "/content/drive/MyDrive/data"
# parquet_dir = os.path.join(base_path, "parquet")
# os.makedirs(parquet_dir, exist_ok=True)

# trans_csv = os.path.join(base_path, "HI-Medium_Trans.csv")
# acc_csv   = os.path.join(base_path, "HI-Medium_accounts.csv")
# patterns_txt = os.path.join(base_path, "HI-Medium_Patterns.txt")

In [ ]:
# # Parquet 저장 폴더
# trans_parquet = os.path.join(parquet_dir, "HI-Medium_Trans.parquet")
# acc_parquet   = os.path.join(parquet_dir, "HI-Medium_accounts.parquet")
# master_parquet = os.path.join(parquet_dir, "HI-Medium_Master.parquet")

In [ ]:
# # (처음 1회만) CSV -> Parquet 변환(스트리밍으로 처리되어 메모리 안전)
# #    - 이미 parquet가 있으면 스킵
# if not os.path.exists(trans_parquet):
#     pl.scan_csv(
#         trans_csv,
#         infer_schema_length=10000,
#         try_parse_dates=False
#     ).sink_parquet(trans_parquet)

# if not os.path.exists(acc_parquet):
#     pl.scan_csv(
#         acc_csv,
#         infer_schema_length=10000,
#     ).sink_parquet(acc_parquet)

In [ ]:
# # Lazy 모드로 CSV 연결
# q_trans = pl.scan_parquet(trans_parquet)
# q_acc   = pl.scan_parquet(acc_parquet)

In [ ]:
# #   Rates LazyFrame
# #   - Currency_Rate.csv를 따로 scan_csv해도 되는데,
# #     지금은 고정 테이블로 만든다고 했으니 lazy DF로 구성
# # -----------------------------
# rates_df = pl.DataFrame({
#     "currency": [
#         "US Dollar","Euro","UK Pound","Swiss Franc","Canadian Dollar",
#         "Australian Dollar","Bitcoin","Saudi Riyal","Shekel","Brazil Real",
#         "Yuan","Mexican Peso","Ruble","Rupee","Yen"
#     ],
#     "rate": [
#         1.0,0.99,1.15,1.03,0.76,
#         0.67,20000.0,0.266,0.29,0.19,
#         0.14,0.05,0.017,0.0125,0.007
#     ]
# }).lazy()

In [ ]:
# # -----------------------------
# #  컬럼명 normalize (데이터마다 Account/Account.1 케이스가 있어서 방어)
# # -----------------------------
# # trans의 실제 컬럼을 먼저 확인하고 싶으면:
# # print(q_trans.columns)  # Lazy에서도 가능

# q_trans = (
#     q_trans
#     # 만약 컬럼명이 'Account', 'Account_duplicated_0'이라고 가정하면:
#     .rename({
#         "Account": "From Account",      # 첫 번째 Account는 보내는 사람
#         "Account_duplicated_0": "To Account"       # 두 번째(중복) Account는 받는 사람
#         # ※ 주의: 위에서 print(q_trans.columns)로 확인한 실제 이름과 다르면
#         #         여기를 실제 이름에 맞춰 수정해야 합니다!
#     })
# )

# # 이제 trans는 최소한 From Account / To Account 컬럼이 있다고 가정하고 진행

In [ ]:
# # -----------------------------
# #  Master build
# #   1) Receiving Currency로 Amount Received USD
# #   2) Payment Currency로 Amount Paid USD
# #   3) accounts를 sender/receiver로 2번 left join (suffix 처리)
# # -----------------------------
# q_master = (
#     q_trans

#     # --- (1) Amount Received USD (Receiving Currency 기준)
#     .join(
#         rates_df.rename({"currency": "Receiving Currency", "rate": "recv_rate"}),
#         on="Receiving Currency",
#         how="left"
#     )
#     .with_columns([
#         (pl.col("recv_rate").fill_null(1.0)).alias("recv_rate"),
#         (pl.col("Amount Received") * pl.col("recv_rate")).alias("Amount_Received_USD")
#     ])

#     # --- (2) Amount Paid USD (Payment Currency 기준)
#     .join(
#         rates_df.rename({"currency": "Payment Currency", "rate": "pay_rate"}),
#         on="Payment Currency",
#         how="left"
#     )
#     .with_columns([
#         (pl.col("pay_rate").fill_null(1.0)).alias("pay_rate"),
#         (pl.col("Amount Paid") * pl.col("pay_rate")).alias("Amount_Paid_USD")
#     ])

#     # --- (3) Sender account join
#     .join(
#         q_acc.select(["Account Number", "Entity Name", "Bank Name"]),
#         left_on="From Account",
#         right_on="Account Number",
#         how="left",
#         suffix="_sender"
#     )
#     .rename({
#         "Entity Name": "Sender_Entity",
#         "Bank Name": "Sender_Bank_Name"
#     })

#     # --- (4) Receiver account join (컬럼 충돌 방지 위해 suffix 사용)
#     .join(
#         q_acc.select(["Account Number", "Entity Name", "Bank Name"]),
#         left_on="To Account",
#         right_on="Account Number",
#         how="left",
#         suffix="_receiver"
#     )
#     .rename({
#         "Entity Name": "Receiver_Entity",
#         "Bank Name": "Receiver_Bank_Name"
#     })

#     # (선택) rate 컬럼 정리하고 싶으면:
#    .drop(["recv_rate","pay_rate"])
# )

In [ ]:
# def parse_aml_patterns(file_path):
#     data = []
#     # 패턴명과 Max Hops/Degree 정보를 추출하기 위한 정규표현식
#     # 예: "BEGIN LAUNDERING ATTEMPT - CYCLE:  Max 12 hops"
#     pattern_re = re.compile(r"BEGIN LAUNDERING ATTEMPT\s*-\s*([^:]+)(?::\s*(.*))?")

#     current_pattern = "NONE"
#     current_meta = ""

#     with open(file_path, 'r', encoding='utf-8') as f:
#         for line in f:
#             line = line.strip()
#             if not line: continue

#             # 1. 패턴 시작점 파싱
#             if "BEGIN LAUNDERING ATTEMPT" in line:
#                 match = pattern_re.search(line)
#                 if match:
#                     current_pattern = match.group(1).strip() # 예: CYCLE
#                     current_meta = match.group(2).strip() if match.group(2) else "" # 예: Max 12 hops
#                 continue

#             # 2. 패턴 종료점
#             if "END LAUNDERING ATTEMPT" in line:
#                 current_pattern = "NONE"
#                 current_meta = ""
#                 continue

#             # 3. 거래 데이터 라인 (CSV 형태)
#             # 형식: Timestamp, From_ID, From_Acc, To_ID, To_Acc, Amount, Currency, ...
#             if current_pattern != "NONE" and "," in line:
#                 parts = line.split(",")
#                 if len(parts) >= 9:
#                     data.append({
#                         "Timestamp": parts[0].strip(),
#                         "From Account": parts[2].strip(), # 814167590 같은 계좌번호
#                         "To Account": parts[4].strip(),
#                         "Amount Received": float(parts[5].strip()),
#                         "Pattern_Type": current_pattern,
#                         "Pattern_Detail": current_meta # Max hops 정보 저장
#                     })

#     return pl.DataFrame(data)

# # --- 실행부 ---
# df_patterns = parse_aml_patterns(patterns_txt)

# # q_master와 조인 (Timestamp, 계좌번호, 금액 3중 키로 정확도 확보)
# # 주의: q_master의 Timestamp와 Amount 타입을 패턴 데이터와 맞춰야 합니다.
# q_master = q_master.join(
#     df_patterns.lazy(),
#     on=["Timestamp", "From Account", "To Account", "Amount Received"],
#     how="left"
# ).with_columns([
#     pl.col("Pattern_Type").fill_null("NONE"),
#     pl.col("Pattern_Detail").fill_null("")
# ])

In [ ]:
# # -----------------------------
# #  sink to Parquet (Streaming write)
# # -----------------------------
# print("Writing master table to Parquet (streaming)...")
# q_master.sink_parquet(master_parquet)
# print(f"Done. Saved: {master_parquet}")

In [ ]:
# master_parquet = "/content/drive/MyDrive/data/parquet/HI-Medium_Master.parquet"

# q_master = pl.scan_parquet(master_parquet)

# 3) 은행 -> 국가 관련 전처리

In [ ]:
# # -----------------------------------------------------------------------------
# # 1. 유니크 은행 명단 추출 (Lazy -> Collect)
# # -----------------------------------------------------------------------------
# # base_path가 정의되어 있다고 가정합니다.
# master_parquet = "/content/drive/MyDrive/data/parquet/HI-Medium_Master.parquet"

# print("1. 유니크 은행 명단 추출 중...")
# q_master = pl.scan_parquet(master_parquet)

# # Sender_Bank_Name만 유니크하게 뽑아옵니다.
# banks = (
#     q_master
#     .select(pl.col("Sender_Bank_Name"))
#     .unique()
#     .collect()
# )

# print(f"   - 총 유니크 은행 수: {len(banks)}")

In [ ]:
# # -----------------------------------------------------------------------------
# # 2. 국가/도시 매핑 사전(Dictionary) 구축 (Lookup Table)
# # -----------------------------------------------------------------------------
# print("2. 매핑 사전(Lookup Table) 구축 중...")

# gc = geonamescache.GeonamesCache()
# countries = gc.get_countries()
# cities = gc.get_cities()

# # 검색용 딕셔너리: { "검색어(소문자)": "표준 국가명" }
# location_map = {}

# # 국가 이름 및 ISO 코드 등록
# for code, data in countries.items():
#     country_name = data['name']
#     location_map[country_name.lower()] = country_name
#     location_map[code.lower()] = country_name # ISO 2자리 코드 (US, KR 등)

# # (pycountry 보완 (공식 명칭 등)
# for country in pycountry.countries:
#     location_map[country.name.lower()] = country.name
#     try:
#         # official_name이 있는 경우
#         if hasattr(country, 'official_name'):
#             location_map[country.official_name.lower()] = country.name
#     except:
#         pass

# # (3) 도시 이름 -> 국가 이름 매핑
# # 동명이인 도시(예: Paris, Texas vs Paris, France) 처리: 인구가 많은 쪽 우선
# city_pop_map = {} # { "city_name": max_population }

# for _, data in cities.items():
#     city_name = data['name'].lower()
#     country_code = data['countrycode']
#     population = data.get('population', 0)

#     # 해당 국가 코드가 우리 국가 사전에 있어야 함
#     if country_code in countries:
#         country_name = countries[country_code]['name']

#         # 이미 등록된 도시라면 인구수 비교해서 갱신
#         if city_name not in city_pop_map or population > city_pop_map[city_name]:
#             city_pop_map[city_name] = population
#             location_map[city_name] = country_name

# # (4) [중요] 사용자 정의 별칭(Alias) 및 예외 처리 등록
# custom_aliases = {
#     "usa": "United States",
#     "us": "United States",
#     "america": "United States",
#     "uk": "United Kingdom",
#     "britain": "United Kingdom",
#     "england": "United Kingdom",
#     "uae": "United Arab Emirates",
#     "korea": "South Korea",
#     "south korea": "South Korea",
#     "russia": "Russia",
#     "vietnam": "Vietnam",
#     "deutschland": "Germany",
#     # (주의) Saudi Arabia와 Crypto는 함수 로직에서 우선 처리하지만 맵에도 추가
#     "saudi": "Saudi Arabia",
#     "arabia": "Saudi Arabia",
# }
# location_map.update(custom_aliases)

In [ ]:
# # -----------------------------------------------------------------------------
# # 3. 국가 추출 함수 (로직 통합 및 강화)
# # -----------------------------------------------------------------------------
# def extract_country_robust(bank_name):
#     if bank_name is None:
#         return "United States" # 기본값

#     # 1. 전처리: 소문자 변환 및 특수문자 제거 후 공백 정리
#     # "Saudi Arabia Bank #123" -> "saudi arabia bank 123"
#     name_clean = re.sub(r"[^a-zA-Z0-9\s]", " ", str(bank_name).lower())
#     name_clean = re.sub(r"\s+", " ", name_clean).strip()

#     # 2. [최우선 규칙] 특정 키워드가 포함되면 즉시 반환 (contains 로직)
#     # Saudi Arabia 처리
#     if "saudi arabia" in name_clean:
#         return "Saudi Arabia"

#     # Crypto 및 오타(Crytpo) 처리
#     if "crypto" in name_clean or "crytpo" in name_clean:
#         return "Crypto"  # 국가명 대신 'Crypto'라는 카테고리로 분류

#     # 3. 토큰 단위 매칭 (단어 쪼개기)
#     tokens = name_clean.split()

#     # (A) "Bank of {Location}" 패턴 확인 (정확도 높음)
#     # 예: Bank of Dallas -> Dallas 검색
#     if len(tokens) >= 3 and tokens[0] == "bank" and tokens[1] == "of":
#         candidate = tokens[2]
#         if candidate in location_map:
#             return location_map[candidate]

#     # (B) 일반 토큰 검색
#     # 문자열의 뒤쪽(도시/국가가 보통 뒤에 옴)부터 검색할 수도 있지만, 여기선 순차 검색
#     for token in tokens:
#         # "Bank", "International" 같은 일반 명사는 제외해야 함 (사전에 없다고 가정하거나 필터링)
#         if token in ["bank", "central", "national", "union", "credit", "unknown"]:
#             continue

#         if token in location_map:
#             return location_map[token]

#     # 4. [Fallback] 아무것도 못 찾았을 경우
#     # 질문 내용상 "나머지 null은 미국 지역은행"이라고 가정하므로 United States 반환
#     return "United States"

In [ ]:
# # -----------------------------------------------------------------------------
# # 4. 데이터프레임에 적용 (map_elements 활용)
# # -----------------------------------------------------------------------------
# print("3. 국가 추출 로직 적용 중...")

# # map_elements로 함수 적용 (return_dtype 지정 필수)
# bank_mapping_df = banks.with_columns(
#     pl.col("Sender_Bank_Name")
#     .map_elements(extract_country_robust, return_dtype=pl.String)
#     .alias("From Country")
# )

In [ ]:
# # -----------------------------------------------------------------------------
# # 5. 검증 (Validation)
# # -----------------------------------------------------------------------------
# print("\n=== 검증 리포트 ===")

# # 1) Null 확인 (로직상 0이어야 함)
# null_count = bank_mapping_df.filter(pl.col("From Country").is_null()).height
# print(f"1. 최종 Null 개수: {null_count} (목표: 0)")

# # 2) Saudi Arabia 확인
# saudi_count = bank_mapping_df.filter(pl.col("From Country") == "Saudi Arabia").height
# print(f"2. Saudi Arabia 추출 개수: {saudi_count}")

# # 3) Crypto (오타 포함) 확인
# crypto_count = bank_mapping_df.filter(pl.col("From Country") == "Crypto").height
# print(f"3. Crypto 추출 개수: {crypto_count}")

# # 4) United States 비율 확인 (Fallback이 잘 작동했는지)
# us_count = bank_mapping_df.filter(pl.col("From Country") == "United States").height
# print(f"4. United States (Fallback 포함) 개수: {us_count}")

In [ ]:
# # -----------------------------------------------------------------------------
# # 6. 원본 데이터와 조인 및 저장
# # -----------------------------------------------------------------------------
# print("\n4. 원본 데이터와 조인 및 저장 중...")

# # LazyFrame으로 변환하여 조인 준비
# mapping_lf = bank_mapping_df.lazy()

# final_lf = (
#     q_master
#     .join(
#         mapping_lf,
#         on="Sender_Bank_Name",
#         how="left"
#     )
# )

# # # 파일 저장 (Parquet)
# # output_path = "q_master_with_country.parquet"
# # final_lf.sink_parquet(output_path)

# # print(f"완료! 파일 저장됨: {output_path}")

In [ ]:
# # # 1. 파일 불러오기 (Lazy)
# # check_lf = pl.scan_parquet("q_master_with_country.parquet")

# # 2. 유니크 은행(Sender_Bank_Name) 기준으로 중복 제거 후 집계
# # 각 국가별로 '몇 종류의 은행'이 있는지 계산합니다.
# bank_dist_report = (
#     final_lf
#     .select(["Sender_Bank_Name", "From Country"])
#     .unique() # <--- 여기서 은행 이름 중복을 제거합니다.
#     .group_by("From Country")
#     .agg(pl.len().alias("unique_bank_count"))
#     .sort("unique_bank_count", descending=True)
#     .collect(engine="streaming")
# )

# print("=== [은행 종류 기준] 국가별 매핑 분포 ===")
# print(bank_dist_report.to_pandas().to_string(index=False))

# # 3. 만약 여전히 null이 있다면, 어떤 은행들이 null인지 상위 20개 확인
# null_banks_sample = (
#     final_lf
#     .select(["Sender_Bank_Name", "From Country"])
#     .unique()
#     .filter(pl.col("From Country").is_null())
#     .collect(engine="streaming")
# )

# print(f"\n매핑되지 않은(null) 유니크 은행 수: {len(null_banks_sample)}개")
# if len(null_banks_sample) > 0:
#     print("--- 매핑 실패 은행 샘플 ---")
#     print(null_banks_sample.head(20))

In [ ]:
# def create_bank_mapping_df(unique_banks_list):
#     """
#     유니크 은행 리스트를 받아 국가 정보를 매핑한 DF 반환 (보정 규칙 포함)
#     """
#     # --- [A] 기초 데이터 로드 ---
#     gc = geonamescache.GeonamesCache()
#     cities = gc.get_cities()
#     city_to_country = {info["name"].upper(): info["countrycode"] for _, info in cities.items()}
#     cc_to_name = {c.alpha_2: c.name for c in pycountry.countries}
#     country_one_tokens = {c.name.upper() for c in pycountry.countries if len(re.findall(r"[A-Z]+", c.name.upper())) == 1}

#     # [수정] ALIASES에 TÜRKIYE와 TURKIYE 추가
#     ALIASES = {
#         "US": "United States", "USA": "United States", "UAE": "United Arab Emirates",
#         "UK": "United Kingdom", "KOREA": "Korea, Republic of", "RUSSIA": "Russian Federation",
#         "TÜRKIYE": "Turkey", "TURKIYE": "Turkey"
#     }

#     def extract_country_internal(bank_name):
#         s = "" if bank_name is None else str(bank_name).upper()
#         toks = re.sub(r"[^A-Z]+", " ", s).strip().split()
#         if not toks: return None
#         # {X} BANK 패턴
#         if len(toks) >= 2 and toks[1] == "BANK":
#             x = toks[0]
#             if x in ALIASES: return ALIASES[x]
#             if x in country_one_tokens: return x.title()
#         # BANK OF {CITY} 패턴
#         if len(toks) >= 3 and toks[0] == "BANK" and toks[1] == "OF":
#             cc = city_to_country.get(toks[2])
#             if cc: return cc_to_name.get(cc, cc)
#         # 전체 토큰 매칭
#         for t in toks:
#             if t in ALIASES: return ALIASES[t]
#             if t in country_one_tokens: return t.title()
#             cc = city_to_country.get(t)
#             if cc: return cc_to_name.get(cc, cc)
#         return None

#     # --- [B] 기초 매핑 생성 ---
#     # 파이썬 리스트를 먼저 생성한 후, 명시적으로 Series로 변환하여 길이를 맞춥니다.
#     # 만약 데이터프레임이 들어오면 첫 번째 컬럼을 리스트로 변환
#     if isinstance(unique_banks_list, pl.DataFrame):
#         input_list = unique_banks_list.to_series(0).to_list()
#     elif isinstance(unique_banks_list, pl.Series):
#         input_list = unique_banks_list.to_list()
#     else:
#         input_list = list(unique_banks_list)

#     # 이제 80179개를 제대로 돕니다.
#     country_raw_list = [extract_country_internal(b) for b in input_list]

#     temp_df = pl.DataFrame({
#         "Bank_Name_Key": input_list,
#         "Country_Raw": pl.Series(country_raw_list, dtype=pl.Utf8)
#     })

#     # --- [C] 규칙 보정 (Saudi Arabia, Crypto, Typos, Turkey) ---
#     name_norm = (
#         pl.col("Bank_Name_Key").cast(pl.Utf8).fill_null("")
#         .str.to_lowercase()
#         .str.replace_all(r"[^a-z0-9]+", " ")
#         .str.strip_chars()
#         .str.replace_all(r"\s+", " ")
#     )

#     # [수정] rules에 türkiye와 turkiye 추가
#     rules = [
#         ("saudi arabia", "Saudi Arabia"),
#         ("crypto", "Crypto"),
#         ("crytpo", "Crypto"),
#         ("türkiye", "Turkey"),
#         ("turkiye", "Turkey")
#     ]

#     # 시작점을 Country_Raw로 잡아서 길이를 맞춤
#     final_expr = pl.col("Country_Raw")

#     # 규칙 표현식 생성
#     for k, v in rules:
#         # 기존 값이 없거나(null), 특정 키워드가 포함된 경우 보정
#         final_expr = (
#             pl.when(name_norm.str.contains(k, literal=True))
#             .then(pl.lit(v))
#             .otherwise(final_expr)
#         )

#     # --- [D] 최종 매핑 테이블 완성 ---
#     bank_mapping_df = temp_df.select([
#         pl.col("Bank_Name_Key").alias("Mapping_Bank_Name"),
#         final_expr
#           .str.replace("Türkiye", "Turkey") # 예외 처리
#           .fill_null("United States")      # 마지막에 US 채우기
#           .alias("Mapped_Country")
#     ])

#     return bank_mapping_df # 할당 연산자(:=) 제거하고 깔끔하게 리턴

# bank_mapping_df = create_bank_mapping_df(banks)  # ✅ 이 줄이 꼭 있어야 함
# bank_mapping_df.head(5)

In [ ]:
# def enrich_master_with_countries(q_master, bank_mapping_df):
#     FATF_MEMBERS = [
#         "Argentina", "Australia", "Austria", "Belgium", "Brazil", "Canada", "China",
#         "Denmark", "Finland", "France", "Germany", "Greece", "Hong Kong", "Iceland",
#         "India", "Ireland", "Israel", "Italy", "Japan", "Korea, Republic of",
#         "Luxembourg", "Malaysia", "Mexico", "Netherlands", "New Zealand", "Norway",
#         "Portugal", "Saudi Arabia", "Singapore", "South Africa", "Spain", "Sweden",
#         "Switzerland", "Turkey", "United Kingdom", "United States"
#     ]

#     # 1. 매핑 테이블 준비
#     s_map = bank_mapping_df.select([
#         pl.col("Mapping_Bank_Name").alias("Sender_Bank_Name"),
#         pl.col("Mapped_Country").alias("From Country")
#     ])

#     r_map = bank_mapping_df.select([
#         pl.col("Mapping_Bank_Name").alias("Receiver_Bank_Name"),
#         pl.col("Mapped_Country").alias("To Country")
#     ])

#     # 2. 조인 수행
#     q_enriched = (
#         q_master
#         .join(s_map.lazy(), on="Sender_Bank_Name", how="left")
#         .join(r_map.lazy(), on="Receiver_Bank_Name", how="left")
#     )

#     # 3. 피쳐 생성 및 보정
#     q_enriched = q_enriched.with_columns([
#         # [수정] 여기서도 혹시 모를 "Türkiye"를 한 번 더 방어적으로 Turkey로 바꿈
#         pl.col("From Country").fill_null("United States").str.replace("Türkiye", "Turkey"),
#         pl.col("To Country").fill_null("United States").str.replace("Türkiye", "Turkey")
#     ]).with_columns([
#         (pl.col("From Country") != pl.col("To Country")).cast(pl.Boolean).alias("IsCrossBorder"),
#         pl.col("From Country").is_in(FATF_MEMBERS).cast(pl.Boolean).alias("IsFromFATF"),
#         pl.col("To Country").is_in(FATF_MEMBERS).cast(pl.Boolean).alias("IsToFATF")
#     ])

#     return q_enriched # 연산 계획이 담긴 LazyFrame 반환

# q_master_enriched = enrich_master_with_countries(q_master, bank_mapping_df)  # ✅

In [ ]:
# # 4. 검증 함수 실행 (이제 'From Country'가 포함된 q_master를 전달합니다)
# def validate_final_mapping(lf):
#     # 컬럼이 있는지 먼저 확인하는 안전 장치 추가
#     cols = lf.collect_schema().names()
#     if "From Country" not in cols:
#         print("❌ 에러: From Country 컬럼이 생성되지 않았습니다. 조인 과정을 확인하세요.")
#         return

#     stats = lf.select([
#         pl.len().alias("Total"),
#         pl.col("From Country").is_null().sum().alias("Nulls"),
#         (pl.col("From Country") == "Saudi Arabia").sum().alias("Saudi_Count"),
#         (pl.col("From Country") == "Crypto").sum().alias("Crypto_Count"),
#         pl.col("IsCrossBorder").sum().alias("CrossBorder_Sum"),
#         pl.col("IsCrossBorder").sum().alias("CrossBorder_Count")
#     ]).collect()

#     print(f"Total Transactions: {stats['Total'][0]:,}")
#     print(f"Remaining Nulls: {stats['Nulls'][0]}")
#     print(f"CrossBorder: {stats['CrossBorder_Count'][0]:,}")
#     print(f"Saudi Arabia Transactions: {stats['Saudi_Count'][0]:,}")
#     print(f"Crypto Transactions: {stats['Crypto_Count'][0]:,}")
#     print(f"Cross-Border Transactions: {stats['CrossBorder_Sum'][0]:,}")

# validate_final_mapping(q_master_enriched)  # ✅ q_master 말고!

In [ ]:
# def _normalize_bank_mapping_df(bank_mapping_df: pl.DataFrame) -> pl.DataFrame:
#     """
#     bank_mapping_df가 어떤 버전이든(스키마가 달라도) 아래 표준 스키마로 통일해서 반환:
#       - Sender_Bank_Name
#       - From Country
#     지원 스키마:
#       A) ["Sender_Bank_Name", "From Country"]
#       B) ["Mapping_Bank_Name", "Mapped_Country"]
#     """
#     cols = set(bank_mapping_df.columns)

#     # 버전 A
#     if {"Sender_Bank_Name", "From Country"}.issubset(cols):
#         out = bank_mapping_df.select([
#             pl.col("Sender_Bank_Name").cast(pl.Utf8).str.strip_chars().alias("Sender_Bank_Name"),
#             pl.col("From Country").cast(pl.Utf8).str.strip_chars().alias("From Country"),
#         ])
#         return out

#     # 버전 B
#     if {"Mapping_Bank_Name", "Mapped_Country"}.issubset(cols):
#         out = bank_mapping_df.select([
#             pl.col("Mapping_Bank_Name").cast(pl.Utf8).str.strip_chars().alias("Sender_Bank_Name"),
#             pl.col("Mapped_Country").cast(pl.Utf8).str.strip_chars().alias("From Country"),
#         ])
#         return out

#     raise ValueError(
#         f"[bank_mapping_df 스키마 인식 실패] columns={bank_mapping_df.columns}\n"
#         "필요 컬럼: (Sender_Bank_Name, From Country) 또는 (Mapping_Bank_Name, Mapped_Country)"
#     )


# def add_aml_features(q_master: pl.LazyFrame, bank_mapping_df: pl.DataFrame) -> pl.LazyFrame:
#     """
#     1) Sender/Receiver 은행명 -> From/To Country 매핑 (bank_mapping_df 버전 A/B 자동 대응)
#     2) Null/Unknown -> United States 보정
#     3) IsCrossBorder, IsFromFATF, IsToFATF 피쳐 생성
#     """

#     FATF_MEMBERS = [
#         "Argentina", "Australia", "Austria", "Belgium", "Brazil", "Canada", "China",
#         "Denmark", "Finland", "France", "Germany", "Greece", "Hong Kong", "Iceland",
#         "India", "Ireland", "Israel", "Italy", "Japan", "Korea, Republic of",
#         "Luxembourg", "Malaysia", "Mexico", "Netherlands", "New Zealand", "Norway",
#         "Portugal", "Saudi Arabia", "Singapore", "South Africa", "Spain", "Sweden",
#         "Switzerland", "Turkey", "United Kingdom", "United States"
#     ]

#     # ✅ bank_mapping_df 어떤 버전이 와도 표준 스키마로 통일
#     mapping_std = _normalize_bank_mapping_df(bank_mapping_df)

#     # sender mapping: Sender_Bank_Name -> From Country
#     sender_map = mapping_std.select([
#         pl.col("Sender_Bank_Name"),
#         pl.col("From Country"),
#     ])

#     # receiver mapping: Receiver_Bank_Name -> To Country
#     receiver_map = mapping_std.select([
#         pl.col("Sender_Bank_Name").alias("Receiver_Bank_Name"),
#         pl.col("From Country").alias("To Country"),
#     ])

#     # ✅ join key도 공백/NULL 방어 (조인 실패 줄이기)
#     q_master = q_master.with_columns([
#         pl.col("Sender_Bank_Name").cast(pl.Utf8).str.strip_chars().alias("Sender_Bank_Name"),
#         pl.col("Receiver_Bank_Name").cast(pl.Utf8).str.strip_chars().alias("Receiver_Bank_Name"),
#     ])

#     # 1) 국가 조인
#     q_master = (
#         q_master
#         .join(sender_map.lazy(), on="Sender_Bank_Name", how="left")
#         .join(receiver_map.lazy(), on="Receiver_Bank_Name", how="left")
#     )

#     # 2) 보정: null / "Unknown" -> United States
#     def _fix_unknown(expr: pl.Expr) -> pl.Expr:
#         return (
#             expr
#             .cast(pl.Utf8)
#             .str.strip_chars()
#             .fill_null("United States")
#             .pipe(lambda e: pl.when(e.str.to_lowercase() == "unknown").then(pl.lit("United States")).otherwise(e))
#         )

#     q_master = q_master.with_columns([
#         _fix_unknown(pl.col("From Country")).alias("From Country"),
#         _fix_unknown(pl.col("To Country")).alias("To Country"),
#     ])

#     # 3) AML 피쳐 생성
#     q_master = q_master.with_columns([
#         (pl.col("From Country") != pl.col("To Country")).cast(pl.UInt8).alias("IsCrossBorder"),
#         pl.col("From Country").is_in(FATF_MEMBERS).cast(pl.UInt8).alias("IsFromFATF"),
#         pl.col("To Country").is_in(FATF_MEMBERS).cast(pl.UInt8).alias("IsToFATF"),
#     ])

#     return q_master

# q_master_enriched = add_aml_features(q_master, bank_mapping_df)

In [ ]:
# def validate_aml_features(q_master_final):
#     print("=== [검증 시작] AML 피쳐 생성 결과 확인 ===")

#     # 1. 기초 통계량 (전체 건수 및 피쳐별 합계)
#     stats = q_master_final.select([
#         pl.len().alias("Total_Rows"),
#         pl.col("IsCrossBorder").sum().alias("CrossBorder_Count"),
#         pl.col("IsFromFATF").sum().alias("FromFATF_Count"),
#         pl.col("IsToFATF").sum().alias("ToFATF_Count")
#     ]).collect()

#     total = stats['Total_Rows'][0]
#     print(f"전체 거래 건수: {total:,}")
#     print(f"해외 송금 건수(CrossBorder): {stats['CrossBorder_Count'][0]:,} ({(stats['CrossBorder_Count'][0]/total*100):.2f}%)")
#     print(f"송신국 FATF 가입 건수: {stats['FromFATF_Count'][0]:,} ({(stats['FromFATF_Count'][0]/total*100):.2f}%)")
#     print(f"수신국 FATF 가입 건수: {stats['ToFATF_Count'][0]:,} ({(stats['ToFATF_Count'][0]/total*100):.2f}%)")

#     # 2. 크로스 체크: 해외 송금인데 사기 거래(Is Laundering == 1)인 비중
#     fraud_cross = q_master_final.filter((pl.col("IsCrossBorder") == 1) & (pl.col("Is Laundering") == 1)).select(pl.len()).collect().item()
#     print(f"해외 송금 중 사기 거래 건수: {fraud_cross:,}")

#     # 3. 샘플 확인
#     print("\n=== 데이터 샘플 (AML 피쳐 중심) ===")
#     print(q_master_final.select([
#         "From Country", "To Country", "IsCrossBorder", "IsFromFATF", "IsToFATF"
#     ]).head(5).collect())

# validate_aml_features(q_master_enriched)

In [ ]:
# def extract_branch_ids_offset(q_master):
#     return q_master.with_columns([
#         # 1. 숫자를 추출합니다 (추출 실패 시 null)
#         pl.col("Sender_Bank_Name")
#         .str.extract(r"#\s*(\d+)", 1)
#         .cast(pl.UInt32)
#         .alias("s_tmp"),

#         pl.col("Receiver_Bank_Name")
#         .str.extract(r"#\s*(\d+)", 1)
#         .cast(pl.UInt32)
#         .alias("r_tmp")
#     ]).with_columns([
#         # 2. 제안하신 로직 적용:
#         # 숫자가 있으면(not null) 값 + 1, 없으면(null) 0
#         pl.when(pl.col("s_tmp").is_not_null())
#           .then(pl.col("s_tmp") + 1)
#           .otherwise(0)
#           .cast(pl.UInt32)
#           .alias("Sender_Bank_Branch_ID"),

#         pl.when(pl.col("r_tmp").is_not_null())
#           .then(pl.col("r_tmp") + 1)
#           .otherwise(0)
#           .cast(pl.UInt32)
#           .alias("Receiver_Bank_Branch_ID")
#     ]).drop(["s_tmp", "r_tmp"]) # 임시 컬럼 삭제

# # 실행
# q_master_enriched = extract_branch_ids_offset(q_master_enriched)

In [ ]:
# # 검증을 위한 샘플 출력
# validation_check = (
#     q_master_enriched
#     .select(["Sender_Bank_Name", "Sender_Bank_Branch_ID"])
#     .unique()
#     .filter(
#         # 세 가지 케이스를 골고루 보기 위한 필터
#         (pl.col("Sender_Bank_Name").str.contains("#0")) |    # 케이스 1: #0 -> 1이 되었는가?
#         (~pl.col("Sender_Bank_Name").str.contains("#")) |    # 케이스 2: #없음 -> 0이 되었는가?
#         (pl.col("Sender_Bank_Name").str.contains("#100"))    # 케이스 3: #100 -> 101이 되었는가?
#     )
#     .head(10)
#     .collect(engine="streaming")
# )

# print("=== 지점 ID 오프셋 적용 결과 ===")
# print(validation_check)

# # 통계 확인
# branch_stats = (
#     q_master_enriched
#     .group_by("Sender_Bank_Branch_ID")
#     .agg(pl.len().alias("count"))
#     .sort("Sender_Bank_Branch_ID")
#     .head(5)
#     .collect(engine="streaming")
# )
# print("\n=== ID별 거래 분포 (0:단일, 1:본점, 2+:지점) ===")
# print(branch_stats)

In [ ]:
# master_parquet = "/content/drive/MyDrive/data/parquet/HI-Medium_Master_v2.parquet"
# print("Writing:", master_parquet)
# q_master_enriched.sink_parquet(master_parquet)
# print("Done!")

In [11]:
master_parquet = "/content/drive/MyDrive/data/parquet/HI-Medium_Master_v2.parquet"

q_master = pl.scan_parquet(master_parquet)

# 4) Timestamp

In [12]:
# =========================================================
# Timestamp 파싱 + 정렬 준비
#   - 원본 Timestamp(string) 유지
#   - ts(Datetime) 새로 생성
#   - ts 파싱 실패(null) 개수 출력
# =========================================================
total_rows = q_master.select(pl.len()).collect().item()

q_master = q_master.with_columns(
    pl.col("Timestamp")
      .str.strptime(pl.Datetime, format="%Y/%m/%d %H:%M", strict=False)
      .alias("ts")
)

null_ts_rows = (
    q_master.filter(pl.col("ts").is_null())
     .select(pl.len())
     .collect()
     .item()
)

valid_rows = total_rows - null_ts_rows

print(f"Total rows                    : {total_rows:,}")
print(f"Rows with NULL ts (parse fail): {null_ts_rows:,}")
print(f"Valid ts rows                 : {valid_rows:,}")

# split은 ts가 유효한 행만 대상으로 진행 (추천)
q_valid = q_master.filter(pl.col("ts").is_not_null())

Total rows                    : 31,898,669
Rows with NULL ts (parse fail): 0
Valid ts rows                 : 31,898,669


In [13]:
# =========================================================
# time-based split 기준(ts cut) 계산 (60:20:20)
#   - collect 최소화: 경계 ts 2개만 뽑음
# =========================================================
n = q_valid.select(pl.len()).collect().item()
train_end_idx = int(n * 0.6)
val_end_idx   = int(n * 0.8)

q_sorted = q_valid.sort("ts")

train_cut_ts = (
    q_sorted.select("ts")
            .slice(train_end_idx, 1)
            .collect()
            .item()
)

val_cut_ts = (
    q_sorted.select("ts")
            .slice(val_end_idx, 1)
            .collect()
            .item()
)

print("Train cutoff ts:", train_cut_ts)
print("Val cutoff ts  :", val_cut_ts)

Train cutoff ts: 2022-09-09 20:43:00
Val cutoff ts  : 2022-09-14 05:46:00


In [14]:
# =========================================================
# split 컬럼 부여 (train/val/test)
# =========================================================
q_split = q_sorted.with_columns(
    pl.when(pl.col("ts") <= train_cut_ts)
      .then(pl.lit("train"))
      .when(pl.col("ts") <= val_cut_ts)
      .then(pl.lit("val"))
      .otherwise(pl.lit("test"))
      .alias("split")
)

# sanity check: split 분포
split_counts = (
    q_split.group_by("split")
           .agg(pl.len().alias("rows"))
           .collect()
)
print(split_counts)

# 라벨 비율 sanity check (optional)
label_counts = (
    q_split.group_by(["split", "Is Laundering"])
           .agg(pl.len().alias("rows"))
           .collect()
)
print(label_counts)

shape: (3, 2)
┌───────┬──────────┐
│ split ┆ rows     │
│ ---   ┆ ---      │
│ str   ┆ u32      │
╞═══════╪══════════╡
│ train ┆ 19139731 │
│ test  ┆ 6378958  │
│ val   ┆ 6379980  │
└───────┴──────────┘
shape: (6, 3)
┌───────┬───────────────┬──────────┐
│ split ┆ Is Laundering ┆ rows     │
│ ---   ┆ ---           ┆ ---      │
│ str   ┆ i64           ┆ u32      │
╞═══════╪═══════════════╪══════════╡
│ val   ┆ 0             ┆ 6370926  │
│ train ┆ 0             ┆ 19124195 │
│ val   ┆ 1             ┆ 9054     │
│ test  ┆ 0             ┆ 6368318  │
│ test  ┆ 1             ┆ 10640    │
│ train ┆ 1             ┆ 15536    │
└───────┴───────────────┴──────────┘


# 6) bucket_ts 생성 (1h)

In [15]:
# =========================================================
# bucket_ts 생성 (ts → 1h truncate)
#   - baseline v1: 거래(row) 단위 모델링이지만,
#     이후 확장을 위해 bucket_ts는 만들어 둠
# =========================================================
q_feat = q_split.with_columns(
    pl.col("ts").dt.truncate("1h").alias("bucket_ts")
).with_columns(
    pl.col("bucket_ts").dt.strftime("%Y-%m-%d %H:%M:%S").alias("bucket_ts_str")
)

# bucket 분포 sanity check (optional)
bucket_check = (
    q_feat.filter(pl.col("split") == "train")
          .select([
              pl.col("bucket_ts").min().alias("train_bucket_min"),
              pl.col("bucket_ts").max().alias("train_bucket_max"),
              pl.len().alias("train_rows"),
          ])
          .collect()
)
print(bucket_check)

shape: (1, 3)
┌─────────────────────┬─────────────────────┬────────────┐
│ train_bucket_min    ┆ train_bucket_max    ┆ train_rows │
│ ---                 ┆ ---                 ┆ ---        │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32        │
╞═════════════════════╪═════════════════════╪════════════╡
│ 2022-09-01 00:00:00 ┆ 2022-09-09 20:00:00 ┆ 19139731   │
└─────────────────────┴─────────────────────┴────────────┘


In [16]:
#“노드 키” 만들기 (송신/수신 각각)
q_feat = q_feat.with_columns([
    (pl.col("From Account").cast(pl.Utf8) + "_" + pl.col("bucket_ts_str").cast(pl.Utf8)).alias("sender_node"),
    (pl.col("To Account").cast(pl.Utf8)   + "_" + pl.col("bucket_ts_str").cast(pl.Utf8)).alias("receiver_node"),
])

In [17]:
# =========================================================
# (6-A) Node indexing: (account_id, bucket_ts) -> node_id
# =========================================================

q_nodes = pl.concat(
    [
        q_feat.select([
            pl.col("sender_node").alias("node_key"),
            pl.col("From Account").cast(pl.Utf8).alias("account_id"),
            pl.col("bucket_ts").alias("bucket_ts"),
            pl.col("split").alias("split_row"),
        ]),
        q_feat.select([
            pl.col("receiver_node").alias("node_key"),
            pl.col("To Account").cast(pl.Utf8).alias("account_id"),
            pl.col("bucket_ts").alias("bucket_ts"),
            pl.col("split").alias("split_row"),
        ]),
    ],
    how="vertical",
).unique(subset=["node_key"])

# node_id 부여 (정렬 후 안정적인 id)
q_nodes = (
    q_nodes
    .sort(["bucket_ts", "account_id"])
    .with_row_index(name="node_id")
)

# 디버깅
n_nodes = q_nodes.select(pl.len()).collect().item()
print("n_nodes =", n_nodes)

# LazyFrame 확보
q_nodes_lf = q_nodes.lazy()

n_nodes = 28756304


In [18]:
# =========================================================
# (6-B) Node label y: predict involvement in NEXT hour (t+1h)
#   y(node at t) = laundering_involvement(node at t+1h)
# =========================================================

# 1) laundering involvement table at time t (current bucket)
q_involve_t = pl.concat(
    [
        q_feat
        .filter(pl.col("Is Laundering") == 1)
        .select([
            pl.col("bucket_ts"),
            pl.col("From Account").cast(pl.Utf8).alias("account_id")
        ]),
        q_feat
        .filter(pl.col("Is Laundering") == 1)
        .select([
            pl.col("bucket_ts"),
            pl.col("To Account").cast(pl.Utf8).alias("account_id")
        ]),
    ],
    how="vertical",
).unique().with_columns(
    pl.lit(1).cast(pl.Int8).alias("involve_t")
)

# 2) shift: (bucket_ts + 1h) 로 옮겨서 "현재 노드의 라벨"로 join
q_involve_next = (
    q_involve_t
    .with_columns((pl.col("bucket_ts") - pl.duration(hours=1)).alias("bucket_ts"))  # ★ 핵심: next를 current로 당김
    .rename({"involve_t": "y"})
)

# 3) node table에 라벨 join (없으면 0)
q_nodes = (
    q_nodes
    .join(q_involve_next, on=["bucket_ts", "account_id"], how="left")
    .with_columns(pl.col("y").fill_null(0).cast(pl.Int64))  # PyTorch loss용 int64
)

# sanity check
print(
    q_nodes.select([
        pl.col("y").sum().alias("pos_nodes"),
        pl.len().alias("total_nodes")
    ]).collect()
)

shape: (1, 2)
┌───────────┬─────────────┐
│ pos_nodes ┆ total_nodes │
│ ---       ┆ ---         │
│ i64       ┆ u32         │
╞═══════════╪═════════════╡
│ 6999      ┆ 28756304    │
└───────────┴─────────────┘


In [19]:
# =========================================================
# (6-C) Node masks by bucket_ts (time-based, no leakage)
#   - train_cut_ts / val_cut_ts 는 Step 4에서 이미 계산됨
#   - node split: bucket_ts 기준
# =========================================================

q_nodes = q_nodes.with_columns(
    pl.when(pl.col("bucket_ts") <= pl.lit(train_cut_ts).dt.truncate("1h"))
      .then(pl.lit("train"))
      .when(pl.col("bucket_ts") <= pl.lit(val_cut_ts).dt.truncate("1h"))
      .then(pl.lit("val"))
      .otherwise(pl.lit("test"))
      .alias("node_split")
)

print(
    q_nodes.group_by("node_split").agg(pl.len().alias("n")).collect()
)
print(
    q_nodes.group_by(["node_split","y"]).agg(pl.len().alias("n")).collect()
)

shape: (3, 2)
┌────────────┬──────────┐
│ node_split ┆ n        │
│ ---        ┆ ---      │
│ str        ┆ u32      │
╞════════════╪══════════╡
│ val        ┆ 5752721  │
│ train      ┆ 17276404 │
│ test       ┆ 5727179  │
└────────────┴──────────┘
shape: (6, 3)
┌────────────┬─────┬──────────┐
│ node_split ┆ y   ┆ n        │
│ ---        ┆ --- ┆ ---      │
│ str        ┆ i64 ┆ u32      │
╞════════════╪═════╪══════════╡
│ test       ┆ 0   ┆ 5725325  │
│ train      ┆ 0   ┆ 17272958 │
│ val        ┆ 1   ┆ 1699     │
│ train      ┆ 1   ┆ 3446     │
│ val        ┆ 0   ┆ 5751022  │
│ test       ┆ 1   ┆ 1854     │
└────────────┴─────┴──────────┘


# 7) 엣지 인덱싱(edge_index) 만들기

In [20]:
# =========================================================
# (7) Build edges: transaction rows -> edge_index (src, dst)
#   - src: sender_node_id
#   - dst: receiver_node_id
# =========================================================

# 1) 거래 row에 sender/receiver node_id 붙이기
q_edges = (
    q_feat
    .join(q_nodes_lf.select(["node_key","node_id"]).rename({"node_key":"sender_node", "node_id":"src_id"}),
          on="sender_node", how="left")
    .join(q_nodes_lf.select(["node_key","node_id"]).rename({"node_key":"receiver_node", "node_id":"dst_id"}),
          on="receiver_node", how="left")
)

# 2) 혹시 매핑 실패(null) 있으면 제거 (이론상 없어야 정상)
q_edges = q_edges.filter(pl.col("src_id").is_not_null() & pl.col("dst_id").is_not_null())

# 3) self-loop 제거(같은 노드로 가는 edge)
q_edges = q_edges.filter(pl.col("src_id") != pl.col("dst_id"))

# edge 수 체크
n_edges = q_edges.select(pl.len()).collect().item()
print("n_edges =", n_edges)

n_edges = 29336701


# 8) 그래프 피처 엔지니어링

In [ ]:
# # =========================================================
# # (7.5) NetworKit Graph Features (1h bucket, prev-hour shift)
# #   - 각 bucket_ts(1h)에 대해 그래프 만들고 중심성 계산
# #   - "prev 1h"를 현재 거래에 붙임 (누수 방지)
# #   - Features (3~5개):
# #       pagerank, core_number, approx_betweenness, degree (optional), weighted_degree(optional)
# # =========================================================

# def build_graph_features_prev1h(
#     q_pl: pl.LazyFrame,
#     bucket_col: str = "bucket_ts",
#     src_col: str = "From Account",
#     dst_col: str = "To Account",
#     weight_col: str = "Amount_Paid_USD",   # 가중치로 쓰고 싶으면 사용
#     sample_max_edges_per_bucket: int | None = None,  # 너무 무거우면 bucket당 edge 샘플링
#     compute_weighted_strength: bool = True,
#     seed: int = 42,
# ) -> pl.DataFrame:
#     """
#     반환: polars DataFrame
#       columns = [bucket_ts, account,
#                  pr, core, abetw, deg, strength]
#     ※ 이 테이블은 '해당 bucket의 그래프' 기준 특징임.
#     ※ 실제 거래에 붙일 때는 bucket_ts를 +1h shift해서 join -> "prev1h" 특징이 됨.
#     """
#     rng = np.random.default_rng(seed)

#     # (1) 필요한 컬럼만 collect (여기서만 pandas로 내림)
#     #     주의: LazyFrame -> collect() 는 비용 큼. (그래도 최소 컬럼만)
#     df = (
#         q_pl.select([bucket_col, src_col, dst_col, weight_col])
#             .collect()
#             .to_pandas()
#     )
#     df[bucket_col] = pd.to_datetime(df[bucket_col])

#     # (2) bucket별 edge list 만들기
#     out_rows = []

#     for b, g in df.groupby(bucket_col, sort=True):
#         if len(g) == 0:
#             continue

#         # 너무 크면 bucket당 edge 샘플링(옵션)
#         if sample_max_edges_per_bucket is not None and len(g) > sample_max_edges_per_bucket:
#             g = g.sample(sample_max_edges_per_bucket, random_state=seed)

#         # ============================
#         # ✅ 여기!!! (그래프 전처리)
#         # ============================
#         g = g[[src_col, dst_col, weight_col]].copy()
#         g[src_col] = g[src_col].astype(str)
#         g[dst_col] = g[dst_col].astype(str)

#         # self-loop 제거 (A → A)
#         g = g[g[src_col] != g[dst_col]]

#         # 중복 edge 제거 (속도 + 안정성)
#         g = g.drop_duplicates(subset=[src_col, dst_col])

#         if len(g) == 0:
#             continue

#         # (3) 노드 매핑 (account -> int)
#         nodes = pd.Index(pd.concat([g[src_col], g[dst_col]], axis=0).astype(str).unique())
#         node2id = {a:i for i,a in enumerate(nodes)}
#         n = len(nodes)

#         # (4) 그래프 구성
#         # 4-1) undirected graph (core, betweenness에 사용)
#         Gu = nk.Graph(n, weighted=False, directed=False)

#         # 4-2) directed graph (pagerank에 사용)
#         Gd = nk.Graph(n, weighted=False, directed=True)

#         # (optional) strength(가중 degree) 계산용 누적
#         strength = np.zeros(n, dtype=np.float64)

#         # edge 추가
#         for _, r in g.iterrows():
#             u = node2id[str(r[src_col])]
#             v = node2id[str(r[dst_col])]

#             # ✅ self-loop 방지 (A -> A)
#             if u == v:
#                 continue

#             Gu.addEdge(u, v)
#             Gd.addEdge(u, v)

#             if compute_weighted_strength:
#                 w = float(r[weight_col]) if pd.notna(r[weight_col]) else 0.0
#                 strength[u] += w
#                 strength[v] += w

#         # ✅ 보험: 혹시 남아있으면 제거
#         Gu.removeSelfLoops()
#         Gd.removeSelfLoops()

#         # (5) 중심성 계산 (가벼운 것들만)
#         # 5-1) PageRank (directed)
#         pr = nk.centrality.PageRank(Gd, damp=0.85, tol=1e-6)
#         pr.run()
#         pr_scores = np.array(pr.scores(), dtype=np.float64)

#         # 5-2) Core number (undirected)
#         core = nk.centrality.CoreDecomposition(Gu)
#         core.run()
#         core_scores = np.array(core.scores(), dtype=np.float64)

#         # 5-3) Approx Betweenness (undirected) - 빠른 근사
#         #     sampleSize 작게 잡으면 속도/품질 trade-off
#         sampleSize = min(200, max(20, int(np.sqrt(n))))

#         try:
#             ab = nk.centrality.ApproxBetweenness(Gu, sampleSize=sampleSize, normalized=True)
#         except TypeError:
#             # fallback: positional args
#             ab = nk.centrality.ApproxBetweenness(Gu, sampleSize, True)

#         ab.run()
#         ab_scores = np.array(ab.scores(), dtype=np.float64)

#         # 5-4) degree (undirected)
#         deg = np.array([Gu.degree(i) for i in range(n)], dtype=np.float64)

#         # (6) bucket 결과를 row로 풀기
#         for acc, i in node2id.items():
#             out_rows.append({
#                 "bucket_ts": b,
#                 "account": acc,
#                 "g_pr": float(pr_scores[i]),
#                 "g_core": float(core_scores[i]),
#                 "g_abetw": float(ab_scores[i]),
#                 "g_deg": float(deg[i]),
#                 "g_strength": float(strength[i]) if compute_weighted_strength else 0.0,
#             })

#     out_df = pd.DataFrame(out_rows)
#     if len(out_df) == 0:
#         return pl.DataFrame({
#             "bucket_ts": [], "account": [],
#             "g_pr": [], "g_core": [], "g_abetw": [], "g_deg": [], "g_strength": []
#         })

#     # polars로 변환
#     return pl.from_pandas(out_df)


# # ---- (A) q_feat은 LazyFrame이 아니고 Lazy/DF 섞여있으니: 지금은 q_feat가 LazyFrame이라고 가정
# # 만약 q_feat가 이미 DataFrame이면: q_feat.lazy()로 넘겨줘도 됨
# graph_node_feat_tbl = build_graph_features_prev1h(
#     q_pl=q_feat.lazy() if isinstance(q_feat, pl.DataFrame) else q_feat,
#     sample_max_edges_per_bucket=200000,      # 너무 무거우면 낮춰 (예: 50000)
#     compute_weighted_strength=True,
# )

# # ---- (B) "prev1h"로 쓰기 위해 bucket_ts를 +1h shift해서 join key로 만든다
# #   - 즉, (b에서 만든 중심성) -> (b+1h 거래에 붙음)
# graph_node_feat_tbl = graph_node_feat_tbl.with_columns(
#     (pl.col("bucket_ts") + pl.duration(hours=1)).alias("bucket_ts_join")
# ).drop("bucket_ts").rename({"bucket_ts_join": "bucket_ts"})

# # ---- (C) sender/receiver 각각에 join해서 붙인다
# #   q_feat에 bucket_ts, From/To Account가 있다고 가정
# graph_node_feat_lf = graph_node_feat_tbl.lazy()

# print(type(q_feat), type(graph_node_feat_tbl))

# q_feat = q_feat.join(
#     graph_node_feat_lf.rename({
#         "account": "From Account",
#         "g_pr": "s_prev1h_pr",
#         "g_core": "s_prev1h_core",
#         "g_abetw": "s_prev1h_abetw",
#         "g_deg": "s_prev1h_deg",
#         "g_strength": "s_prev1h_strength",
#     }),
#     on=["bucket_ts", "From Account"],
#     how="left"
# ).join(
#     graph_node_feat_lf.rename({
#         "account": "To Account",
#         "g_pr": "r_prev1h_pr",
#         "g_core": "r_prev1h_core",
#         "g_abetw": "r_prev1h_abetw",
#         "g_deg": "r_prev1h_deg",
#         "g_strength": "r_prev1h_strength",
#     }),
#     on=["bucket_ts", "To Account"],
#     how="left"
# )

# # ---- (D) 결측 채우기: 이전 1시간 그래프에 등장 안 한 계정이면 0이 자연스러움
# GFEATS = [
#     "s_prev1h_pr","s_prev1h_core","s_prev1h_abetw","s_prev1h_deg","s_prev1h_strength",
#     "r_prev1h_pr","r_prev1h_core","r_prev1h_abetw","r_prev1h_deg","r_prev1h_strength",
# ]
# q_feat = q_feat.with_columns([pl.col(c).fill_null(0) for c in GFEATS])

# # ---- (E) 타입 정리(선택)
# q_feat = q_feat.with_columns([pl.col(c).cast(pl.Float32) for c in GFEATS])

In [ ]:
# q_feat.select([
#     "bucket_ts","split","From Account","To Account",
#     "s_prev1h_pr","s_prev1h_core","s_prev1h_abetw",
#     "r_prev1h_pr","r_prev1h_core","r_prev1h_abetw",
# ]).head(5).collect()

In [ ]:
# # =========================================================
# # (8) Window Graph Features (LIGHT v0)
# #   - 목적: "그래프 느낌" 나는 피처를 아주 싸게 추가
# #   - 구현: Polars rolling/groupby 기반
# #   - 누수 방지: shift(1)로 '현재 거래 이전'만 사용
# #   - 정의:
# #       deg_out = 최근 윈도우에서 distinct receiver 수
# #       deg_in  = 최근 윈도우에서 distinct sender 수
# #       wout    = 최근 윈도우에서 총 송금액(Amount_Paid_USD)
# #       win     = 최근 윈도우에서 총 수금액(Amount_Received_USD)
# # =========================================================

# # 안전: 정렬 (rolling_*_by는 ts 정렬이 중요)
# q_feat = q_feat.sort(["From Account", "To Account", "ts"])

# # # helper: distinct neighbor (out-degree like)
# # # sender 기준: From Account의 "최근 window"에서 To Account distinct count
# # q_feat = q_feat.with_columns([
# #     # ----- sender out-degree (distinct receivers) -----
# #     pl.col("To Account")
# #       .cast(pl.Utf8)
# #       .rolling_n_unique_by("ts", window_size="1h")
# #       .shift(1).over("From Account")
# #       .alias("s_deg_out_1h"),

# #     pl.col("To Account")
# #       .cast(pl.Utf8)
# #       .rolling_n_unique_by("ts", window_size="24h")
# #       .shift(1).over("From Account")
# #       .alias("s_deg_out_24h"),

# #     # ----- sender in-degree (distinct senders into this sender) -----
# #     # "sender가 receiver로 등장"하는 케이스를 엄밀히 하려면
# #     # To Account 기준으로 From Account unique를 봐야 해서, 아래 receiver 블록에서 같이 계산함.
# # ])

# # # receiver 쪽 rolling은 To Account 기준 정렬이 안전
# # q_feat = q_feat.sort(["To Account", "ts"])

# # q_feat = q_feat.with_columns([
# #     # ----- receiver in-degree (distinct senders) -----
# #     pl.col("From Account")
# #       .cast(pl.Utf8)
# #       .rolling_n_unique_by("ts", window_size="1h")
# #       .shift(1).over("To Account")
# #       .alias("r_deg_in_1h"),

# #     pl.col("From Account")
# #       .cast(pl.Utf8)
# #       .rolling_n_unique_by("ts", window_size="24h")
# #       .shift(1).over("To Account")
# #       .alias("r_deg_in_24h"),

# #     # ----- receiver out-degree (distinct receivers this receiver sent to) -----
# #     # receiver가 sender로 등장하는 케이스를 엄밀히 하려면 From Account 기준 To Account unique가 필요
# #     # 이미 sender out-degree가 있으니, 대칭 피처까지 강제하지 않고 v0에서는 생략해도 됨.
# # ])

# # amount-weighted in/out (strength)
# # sender 기준은 From Account로, receiver 기준은 To Account로 rolling sum
# q_feat = q_feat.sort(["From Account", "ts"])
# q_feat = q_feat.with_columns([
#     pl.col("Amount_Paid_USD")
#       .rolling_sum_by("ts", window_size="1h")
#       .shift(1).over("From Account")
#       .alias("s_wout_1h"),

#     pl.col("Amount_Paid_USD")
#       .rolling_sum_by("ts", window_size="24h")
#       .shift(1).over("From Account")
#       .alias("s_wout_24h"),

#     pl.col("Amount_Received_USD")
#       .rolling_sum_by("ts", window_size="1h")
#       .shift(1).over("From Account")
#       .alias("s_win_1h"),

#     pl.col("Amount_Received_USD")
#       .rolling_sum_by("ts", window_size="24h")
#       .shift(1).over("From Account")
#       .alias("s_win_24h"),
# ])

# q_feat = q_feat.sort(["To Account", "ts"])
# q_feat = q_feat.with_columns([
#     pl.col("Amount_Received_USD")
#       .rolling_sum_by("ts", window_size="1h")
#       .shift(1).over("To Account")
#       .alias("r_win_1h"),

#     pl.col("Amount_Received_USD")
#       .rolling_sum_by("ts", window_size="24h")
#       .shift(1).over("To Account")
#       .alias("r_win_24h"),

#     pl.col("Amount_Paid_USD")
#       .rolling_sum_by("ts", window_size="1h")
#       .shift(1).over("To Account")
#       .alias("r_wout_1h"),

#     pl.col("Amount_Paid_USD")
#       .rolling_sum_by("ts", window_size="24h")
#       .shift(1).over("To Account")
#       .alias("r_wout_24h"),
# ])

# # ---- null fill (첫 거래/윈도우 초기구간은 null -> 0)
# GRAPH_WIN_FEATS = [
#     # "s_deg_out_1h","s_deg_out_24h",
#     # "r_deg_in_1h","r_deg_in_24h",
#     "s_wout_1h","s_wout_24h","s_win_1h","s_win_24h",
#     "r_win_1h","r_win_24h","r_wout_1h","r_wout_24h",
# ]
# GRAPH_WIN_FEATS = [c for c in GRAPH_WIN_FEATS if c in q_feat.columns]

# q_feat = q_feat.with_columns([pl.col(c).fill_null(0) for c in GRAPH_WIN_FEATS])

# # (optional) 타입 정리 (트리모델에서 float32가 깔끔)
# q_feat = q_feat.with_columns([
#     pl.col(c).cast(pl.Float32) if q_feat.schema[c] in (pl.Float64,) else pl.col(c)
#     for c in GRAPH_WIN_FEATS
# ])

In [ ]:
# q_feat.select(
#     ["ts","From Account","To Account"] + GRAPH_WIN_FEATS
# ).head(5).collect()

In [ ]:
# # =========================================================
# # (8-1.6) Edge(pair) Features (LIGHT v0)
# #   - A->B pair(From Account, To Account) 기준
# #   - 1h/24h window에서 "이 pair가 얼마나 자주/얼마나 크게" 발생했는지
# #   - 누수 방지: shift(1)
# #   - 5개:
# #       e_cnt_1h, e_sum_paid_1h
# #       e_cnt_24h, e_sum_paid_24h
# #       e_new_pair_flag (train split 기준: 처음 보는 pair면 True)
# # =========================================================

# # (0) pair key (polars struct로 grouping)
# PAIR_KEY = ["From Account", "To Account"]

# # (1) rolling은 ts 정렬 중요
# q_feat = q_feat.sort(PAIR_KEY + ["ts"])

# # (2) past window: count / sum (Amount_Paid_USD 기준)
# q_feat = q_feat.with_columns([
#     pl.lit(1).cast(pl.Int32)
#       .rolling_sum_by("ts", window_size="1h")
#       .shift(1).over(PAIR_KEY)
#       .alias("e_cnt_1h_past"),

#     pl.col("Amount_Paid_USD")
#       .rolling_sum_by("ts", window_size="1h")
#       .shift(1).over(PAIR_KEY)
#       .alias("e_sum_paid_1h_past"),

#     pl.lit(1).cast(pl.Int32)
#       .rolling_sum_by("ts", window_size="24h")
#       .shift(1).over(PAIR_KEY)
#       .alias("e_cnt_24h_past"),

#     pl.col("Amount_Paid_USD")
#       .rolling_sum_by("ts", window_size="24h")
#       .shift(1).over(PAIR_KEY)
#       .alias("e_sum_paid_24h_past"),
# ])

# # (3) 누수 방지 new_pair_flag:
# #     "train에서 한 번도 등장하지 않은 pair가 val/test에 등장하면 True"
# #     -> 운영 관점에서 의미 있고, 라벨 누수도 없음.
# #     -> 구현: train에서 pair set 만들고, 전체에 join

# # train에서 unique pair 목록
# q_train_pairs = (
#     q_feat
#     .filter(pl.col("split") == "train")
#     .select(PAIR_KEY)
#     .unique()
#     .with_columns(pl.lit(1).alias("seen_in_train"))
# )

# # 전체에 left join 후, null이면 unseen -> new_pair_flag = True
# q_feat = (
#     q_feat
#     .join(q_train_pairs, on=PAIR_KEY, how="left")
#     .with_columns(
#         pl.col("seen_in_train").is_null().alias("e_new_pair_flag")
#     )
#     .drop(["seen_in_train"])
# )

# # (4) null fill (초기 window는 null -> 0)
# EDGE_FEATS = [
#     "e_cnt_1h_past","e_sum_paid_1h_past",
#     "e_cnt_24h_past","e_sum_paid_24h_past",
#     "e_new_pair_flag",
# ]
# EDGE_FEATS = [c for c in EDGE_FEATS if c in q_feat.columns]

# q_feat = q_feat.with_columns([
#     pl.col("e_cnt_1h_past").fill_null(0),
#     pl.col("e_sum_paid_1h_past").fill_null(0),
#     pl.col("e_cnt_24h_past").fill_null(0),
#     pl.col("e_sum_paid_24h_past").fill_null(0),
#     pl.col("e_new_pair_flag").fill_null(False),
# ])

# # (optional) 타입 정리
# q_feat = q_feat.with_columns([
#     pl.col("e_sum_paid_1h_past").cast(pl.Float32),
#     pl.col("e_sum_paid_24h_past").cast(pl.Float32),
#     pl.col("e_cnt_1h_past").cast(pl.Int32),
#     pl.col("e_cnt_24h_past").cast(pl.Int32),
#     pl.col("e_new_pair_flag").cast(pl.Boolean),
# ])

In [ ]:
# q_feat.select(
#     ["ts","split","From Account","To Account",
#      "e_cnt_1h_past","e_sum_paid_1h_past",
#      "e_cnt_24h_past","e_sum_paid_24h_past",
#      "e_new_pair_flag"]
# ).head(5).collect()

# 8-1) 기존 피처 엔지니어링

In [21]:
# -------------------------
# (A) 상수/룩업 리스트
# -------------------------
PAYMENT_FORMATS = ["ACH", "Cheque", "Bitcoin", "Cash", "Credit Card", "Wire", "Reinvestment"]

HIGH_RISK_SENDER_BANKS = [
    "National Bank of Dallas", "Savings Bank of Augusta", "China Bank #27",
    "India Bank #96", "Brazil Bank #128", "National Bank of Milford",
    "Savings Bank of Sacramento", "Saudi Arabia Bank #14", "Israel Bank #16",
    "Golden Credit Union"
]

HIGH_RISK_RECEIVER_BANKS = [
    "China Bank #292", "China Bank #27", "China Bank #22", "Japan Bank #143",
    "Brazil Bank #50", "Bank of Denver", "Saudi Arabia Bank #56",
    "Israel Bank #48", "The Pine Bank", "National Bank of Milford"
]

# 엔티티 타입 가중치 (v1: 단순 룰)
ENTITY_TYPE_SCORE_MAP = {
    "Corporation": 2.0,
    "Sole Proprietorship": 1.5
}

# Payment method risk (v1: 단순 룰)
# (정확한 가중치는 EDA에서 조정 가능. 지금은 “비중 높았던 수단에 가중”만 구현)
PAYMENT_RISK_MAP = {
    "ACH": 3.0,
    "Bitcoin": 2.5,
    "Cheque": 2.5,
    "Cash": 1.5,
    "Credit Card": 1.3,
    "Wire": 1.0,
    "Reinvestment": 1.0
}

In [22]:
# -------------------------
# (B) Timestamp 파생 + (추가) hour_sin/cos + dow_cat
#  - hour/day_of_week는 유지(다른 코드 깨짐 방지)
#  - hour_sin/cos: 주기성 반영
#  - dow_cat: 요일을 "순서 없는 범주"로 명시 (필요시 XGBoost categorical용)
# -------------------------
q_feat = q_feat.with_columns([
    # 기존 파생 (그대로 유지)
    pl.col("ts").dt.hour().cast(pl.Int16).alias("hour"),
    pl.col("ts").dt.weekday().cast(pl.Int8).alias("day_of_week"),  # 월=1 ~ 일=7
    pl.col("ts").dt.weekday().is_in([6, 7]).alias("is_weekend"),
    pl.col("ts").dt.date().alias("ts_day"),

    # 새벽 여부 (예: 0~5시)
    pl.col("ts").dt.hour().is_between(0, 5).alias("is_dawn"),

    # TimeofDay bucket (예시: 새벽/오전/오후/밤)
    pl.when(pl.col("ts").dt.hour().is_between(0, 5)).then(pl.lit("dawn"))
      .when(pl.col("ts").dt.hour().is_between(6, 11)).then(pl.lit("morning"))
      .when(pl.col("ts").dt.hour().is_between(12, 17)).then(pl.lit("afternoon"))
      .otherwise(pl.lit("evening"))
      .alias("timeofday_bucket"),

]).with_columns([
    # 추가: Hour 주기성(sin/cos)
    (pl.col("hour").cast(pl.Float32) * (2 * np.pi / 24)).sin().alias("hour_sin"),
    (pl.col("hour").cast(pl.Float32) * (2 * np.pi / 24)).cos().alias("hour_cos"),

    # 추가: 요일을 범주형으로(순서 제거)
    pl.col("day_of_week").cast(pl.Utf8).cast(pl.Categorical).alias("dow_cat"),
])

In [23]:
# -------------------------
# (C) Amount 파생 (float32 + log)
# -------------------------
q_feat = q_feat.with_columns([
    # float32 캐스팅
    pl.col("Amount_Paid_USD").cast(pl.Float32),
    pl.col("Amount_Received_USD").cast(pl.Float32),

    # log1p
    pl.col("Amount_Paid_USD").log1p().alias("log_amount_paid_usd"),
    pl.col("Amount_Received_USD").log1p().alias("log_amount_received_usd"),

    # Round number (1000 단위)
    (pl.col("Amount_Paid_USD") % 1000 == 0).alias("is_round_1000_paid"),
    (pl.col("Amount_Received_USD") % 1000 == 0).alias("is_round_1000_received"),

    # 더 강한 라운드 (10000 단위)
    (pl.col("Amount_Paid_USD") % 10000 == 0).alias("is_round_10000_paid"),
])

In [25]:
# -------------------------
# (D) Payment Format 원핫 + 파생
# -------------------------
# 원핫(각 format별 bool)
q_feat = q_feat.with_columns([
    (pl.col("Payment Format") == "ACH").alias("is_ach"),
    (pl.col("Payment Format") == "Cheque").alias("is_cheque"),
    (pl.col("Payment Format") == "Bitcoin").alias("is_bitcoin_fmt"),
    (pl.col("Payment Format") == "Cash").alias("is_cash"),
    (pl.col("Payment Format") == "Credit Card").alias("is_credit_card"),
    (pl.col("Payment Format") == "Wire").alias("is_wire"),
    (pl.col("Payment Format") == "Reinvestment").alias("is_reinvestment"),
])

# Crypto 여부 (format이 Bitcoin이거나 currency가 Bitcoin이면 True)
q_feat = q_feat.with_columns([
    ((pl.col("Payment Format") == "Bitcoin") | (pl.col("Payment Currency") == "Bitcoin"))
      .alias("is_crypto_transfer"),

    # High value ACH (>= 1,000,000 USD & ACH)
    ((pl.col("Payment Format") == "ACH") & (pl.col("Amount_Paid_USD") >= 1_000_000))
      .alias("is_high_value_ach"),
])

# Payment_Method_Risk (format 기반 가중치)
q_feat = q_feat.with_columns([
    pl.col("Payment Format")
      .replace(PAYMENT_RISK_MAP, default=1.0)
      .cast(pl.Float32)
      .alias("payment_method_risk"),
])

# format × amount interaction (v1용)
q_feat = q_feat.with_columns([
    (pl.col("payment_method_risk") * pl.col("log_amount_paid_usd")).alias("risk_x_log_paid"),
    (pl.col("is_ach").cast(pl.Int8) * pl.col("log_amount_paid_usd")).alias("ach_x_log_paid"),
    (pl.col("is_bitcoin_fmt").cast(pl.Int8) * pl.col("log_amount_paid_usd")).alias("btc_x_log_paid"),
])

In [26]:
# -------------------------
# (E) Entity type 파싱 + 점수
# -------------------------
# "Corporation #26522" → "Corporation"
q_feat = q_feat.with_columns([
    pl.col("Sender_Entity")
      .cast(pl.Utf8)
      .str.replace(r"\s*#\d+.*$", "")   # "#숫자" 이후 제거
      .alias("sender_entity_type"),

    pl.col("Receiver_Entity")
      .cast(pl.Utf8)
      .str.replace(r"\s*#\d+.*$", "")
      .alias("receiver_entity_type"),
])

# 타입 점수
q_feat = q_feat.with_columns([
    pl.col("sender_entity_type")
      .replace(ENTITY_TYPE_SCORE_MAP, default=1.0)
      .cast(pl.Float32)
      .alias("sender_entity_type_score"),

    pl.col("receiver_entity_type")
      .replace(ENTITY_TYPE_SCORE_MAP, default=1.0)
      .cast(pl.Float32)
      .alias("receiver_entity_type_score"),
])


In [24]:
# -------------------------
# (F) bank 관련
# -------------------------
q_feat = q_feat.with_columns([
    pl.col("Sender_Bank_Name").is_in(HIGH_RISK_SENDER_BANKS).alias("high_risk_sender_bank_flag"),
    pl.col("Receiver_Bank_Name").is_in(HIGH_RISK_RECEIVER_BANKS).alias("high_risk_receiver_bank_flag"),
])

q_feat = q_feat.with_columns([
    pl.col("From Bank").cast(pl.Utf8).cast(pl.Categorical),
    pl.col("To Bank").cast(pl.Utf8).cast(pl.Categorical),
])

q_feat = q_feat.with_columns([
    pl.col("Sender_Bank_Branch_ID").cast(pl.Utf8).cast(pl.Categorical),
    pl.col("Receiver_Bank_Branch_ID").cast(pl.Utf8).cast(pl.Categorical),
])

In [27]:
# 1) 거래 row에 sender/receiver node_id 붙이기
q_edges = (
    q_feat
    .join(q_nodes_lf.select(["node_key","node_id"]).rename({"node_key":"sender_node", "node_id":"src_id"}),
          on="sender_node", how="left")
    .join(q_nodes_lf.select(["node_key","node_id"]).rename({"node_key":"receiver_node", "node_id":"dst_id"}),
          on="receiver_node", how="left")
)

# 2) 혹시 매핑 실패(null) 있으면 제거 (이론상 없어야 정상)
q_edges = q_edges.filter(pl.col("src_id").is_not_null() & pl.col("dst_id").is_not_null())

# 3) self-loop 제거(같은 노드로 가는 edge)
q_edges = q_edges.filter(pl.col("src_id") != pl.col("dst_id"))

In [28]:
# =========================================================
# (7-B) Edge attributes (edge_attr) - minimal, stable
#   - 이미 Step 8-1에서 만든 컬럼이 있으면 재사용
# =========================================================

EDGE_ATTR_COLS = [
    "is_weekend",
    # "ts_day",
    "is_dawn",
    # "timeofday_bucket",
    "hour_sin", "hour_cos",
    # "Amount_Paid_USD",
    # "Amount_Received_USD",
    "log_amount_paid_usd",
    "log_amount_received_usd",
    # "is_round_1000_paid",
    # "is_round_1000_received",
    # "is_round_10000_paid",
    "is_ach",
    # "is_cheque",
    "is_bitcoin_fmt",
    # "is_cash",
    # "is_credit_card",
    # "is_wire",
    # "is_reinvestment",
    "is_crypto_transfer",
    "is_high_value_ach",
    "payment_method_risk",
    "risk_x_log_paid",
    "ach_x_log_paid",
    "btc_x_log_paid",
    "sender_entity_type_score",
    "receiver_entity_type_score",
    # "high_risk_sender_bank_flag",
    # "high_risk_receiver_bank_flag",
    "From Country",
    "To Country",
    "IsCrossBorder",
    "IsFromFATF",
    "IsToFATF",
]

# 없는 컬럼은 자동 제외(코드 깨짐 방지)
EDGE_ATTR_COLS = [c for c in EDGE_ATTR_COLS if c in q_edges.columns]

# bool -> float32로 통일 (PyTorch tensor 안정)
q_edges = q_edges.with_columns([
    (pl.col(c).cast(pl.Float32) if q_edges.schema[c] != pl.Float32 else pl.col(c))
    for c in EDGE_ATTR_COLS
])

# null -> 0
q_edges = q_edges.with_columns([pl.col(c).fill_null(0) for c in EDGE_ATTR_COLS])

print("edge_attr cols =", EDGE_ATTR_COLS)

edge_attr cols = ['is_weekend', 'is_dawn', 'hour_sin', 'hour_cos', 'log_amount_paid_usd', 'log_amount_received_usd', 'is_ach', 'is_bitcoin_fmt', 'is_crypto_transfer', 'is_high_value_ach', 'payment_method_risk', 'risk_x_log_paid', 'ach_x_log_paid', 'btc_x_log_paid', 'sender_entity_type_score', 'receiver_entity_type_score', 'From Country', 'To Country', 'IsCrossBorder', 'IsFromFATF', 'IsToFATF']


In [29]:
EDGE_CAT_COLS = [
    "dow_cat",
    "timeofday_bucket",
    "Payment Currency",
    "Receiving Currency",
    "Sender_Bank_Name",
    "Receiver_Bank_Name",
    "sender_entity_type",
    "receiver_entity_type",
    "From Bank",
    "To Bank",
]
EDGE_CAT_COLS = [c for c in EDGE_CAT_COLS if c in q_edges.columns]

In [ ]:
# q_edges는 이미 split 컬럼을 갖고 있음 (train/val/test)

cat_maps = {}      # col -> {category_str: idx}
cat_sizes = {}     # col -> vocab_size (including UNK=0)

for c in EDGE_CAT_COLS:
    uniq = (
        q_edges
        .filter(pl.col("split") == "train")
        .select(pl.col(c).cast(pl.Utf8))
        .unique()
        .drop_nulls()
        .collect()
        .to_series()
        .to_list()
    )
    # UNK=0, 나머지 1..K
    cat_maps[c] = {v: i+1 for i, v in enumerate(uniq)}
    cat_sizes[c] = len(uniq) + 1

In [ ]:
# c_id 컬럼 생성 (string->int), 없는 값/NULL은 0
for c in EDGE_CAT_COLS:
    q_edges = q_edges.with_columns(
        pl.col(c).cast(pl.Utf8).replace(cat_maps[c], default=0).cast(pl.Int64).alias(f"{c}__id")
    )

EDGE_CAT_ID_COLS = [f"{c}__id" for c in EDGE_CAT_COLS]
print("edge_cat cols =", EDGE_CAT_COLS)
print("edge_cat id cols =", EDGE_CAT_ID_COLS)

In [ ]:
# # 확인하고 싶은 핵심 피처만 뽑아서 head 5
# inspect_cols = [
#     # timestamp 관련
#     "Timestamp", "ts", "hour", "day_of_week", "is_weekend", "is_dawn", "timeofday_bucket",

#     # amount 관련
#     "Amount_Paid_USD", "log_amount_paid_usd",
#     "Amount_Received_USD", "log_amount_received_usd",
#     "is_round_1000_paid",

#     # payment format
#     "Payment Format",
#     "is_ach", "is_bitcoin_fmt", "is_crypto_transfer",
#     "is_high_value_ach", "payment_method_risk", "risk_x_log_paid",

#     # entity / bank
#     "Sender_Entity", "sender_entity_type", "sender_entity_type_score",
#     "Sender_Bank_Name", "high_risk_sender_bank_flag",

#     # label / split
#     "Is Laundering", "split"
# ]

# q_feat.select(inspect_cols).head(5).collect()

# 8-2) groupby 필요한 피처 엔지니어링

In [ ]:
# ---------------------------------------------------------
# 0) 준비: _one 만들고, 정렬 (rolling_*_by + over는 정렬 중요)
# ---------------------------------------------------------
q_feat = q_feat.with_columns(pl.lit(1).cast(pl.Int32).alias("_one"))

# sender/receiver 둘 다 만들 거라서, 최소 ts 정렬 + 계정별 정렬
# (sender 만들 땐 From Account 기준, receiver 만들 땐 To Account 기준 정렬이 이상적)
# 여기서는 한 번에 가려고 1) From Account 2) To Account 3) ts 로 정렬
q_feat = q_feat.sort(["From Account", "ts"])


# ---------------------------------------------------------
# 1) Sender(From Account) - 1h / 24h
# ---------------------------------------------------------
q_feat = q_feat.with_columns([
    # 1h: count
    pl.col("_one")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_cnt_1h_past"),

    # 1h: Amount_Paid_USD sum/mean/max/std
    pl.col("Amount_Paid_USD")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_sum_paid_1h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_mean_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_mean_paid_1h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_max_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_max_paid_1h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_std_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_std_paid_1h_past"),

    # 1h: Amount_Received_USD sum/mean
    pl.col("Amount_Received_USD")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_sum_recv_1h_past"),

    pl.col("Amount_Received_USD")
      .rolling_mean_by("ts", window_size="1h")
      .shift(1).over("From Account")
      .alias("s_mean_recv_1h_past"),

    # 24h: count, paid sum
    pl.col("_one")
      .rolling_sum_by("ts", window_size="24h")
      .shift(1).over("From Account")
      .alias("s_cnt_24h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_sum_by("ts", window_size="24h")
      .shift(1).over("From Account")
      .alias("s_sum_paid_24h_past"),
])

# sender net (1h)
q_feat = q_feat.with_columns(
    (pl.col("s_sum_recv_1h_past") - pl.col("s_sum_paid_1h_past")).alias("s_net_1h_past")
)


# ---------------------------------------------------------
# 2) Receiver(To Account) - 1h / 24h
# ---------------------------------------------------------
# receiver 쪽은 To Account 기준으로 rolling이므로,
# 엄밀히는 q_feat = q_feat.sort(["To Account","ts"])가 가장 안전함.
# 이미 위에서 ["From Account","To Account","ts"]로 정렬했지만,
# receiver 안정성 위해 한 번 더 정렬 권장:
q_feat = q_feat.sort(["To Account", "ts"])

q_feat = q_feat.with_columns([
    # 1h: count
    pl.col("_one")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_cnt_1h_past"),

    # 1h: Amount_Received_USD sum/mean/max/std
    pl.col("Amount_Received_USD")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_sum_recv_1h_past"),

    pl.col("Amount_Received_USD")
      .rolling_mean_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_mean_recv_1h_past"),

    pl.col("Amount_Received_USD")
      .rolling_max_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_max_recv_1h_past"),

    pl.col("Amount_Received_USD")
      .rolling_std_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_std_recv_1h_past"),

    # 1h: Amount_Paid_USD sum/mean
    pl.col("Amount_Paid_USD")
      .rolling_sum_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_sum_paid_1h_past"),

    pl.col("Amount_Paid_USD")
      .rolling_mean_by("ts", window_size="1h")
      .shift(1).over("To Account")
      .alias("r_mean_paid_1h_past"),

    # 24h: count
    pl.col("_one")
      .rolling_sum_by("ts", window_size="24h")
      .shift(1).over("To Account")
      .alias("r_cnt_24h_past"),
])

# receiver net (1h)
q_feat = q_feat.with_columns(
    (pl.col("r_sum_recv_1h_past") - pl.col("r_sum_paid_1h_past")).alias("r_net_1h_past")
)

# ---------------------------------------------------------
# 3) TODO: n_unique (s_nuniq_to_24h_past / r_nuniq_from_24h_past)
# ---------------------------------------------------------
# rolling_n_unique_by가 없어서, 이 2개는 "rolling_*_by"만으로는 바로 못 만들고,
# 다른 방식(예: groupby_dynamic + join, 또는 window id로 explode/agg 등) 필요.
# 일단은 여기서는 제외.


In [ ]:
PAST_FILL_ZERO = [
    # sender 10(+net optional)
    "s_cnt_1h_past",
    "s_sum_paid_1h_past",
    "s_mean_paid_1h_past",
    "s_max_paid_1h_past",
    "s_std_paid_1h_past",
    "s_sum_recv_1h_past",
    "s_mean_recv_1h_past",
    "s_cnt_24h_past",
    "s_sum_paid_24h_past",
    # "s_nuniq_to_24h_past",
    "s_net_1h_past",  # 만들었으면 포함

    # receiver 9
    "r_cnt_1h_past",
    "r_sum_recv_1h_past",
    "r_mean_recv_1h_past",
    "r_max_recv_1h_past",
    "r_std_recv_1h_past",
    "r_sum_paid_1h_past",
    "r_mean_paid_1h_past",
    "r_cnt_24h_past",
    # "r_nuniq_from_24h_past",
    "r_net_1h_past",  # 만들었으면 포함
]

q_feat = q_feat.with_columns([
    pl.col(c).fill_null(0)
    for c in PAST_FILL_ZERO
])

In [ ]:
# =========================================================
# (8-2-GNN) Promote tabular aggregates -> node features
#   - row feature(From/To 기준) -> node(account,bucket)로 집계
#   - v1 최소만: count/sum 정도
# =========================================================

NODE_AGG_COLS_SENDER = [c for c in ["s_cnt_1h_past","s_sum_paid_1h_past","s_mean_paid_1h_past","s_max_paid_1h_past","s_std_paid_1h_past","s_sum_recv_1h_past","s_mean_recv_1h_past","s_cnt_24h_past","s_sum_paid_24h_past","s_net_1h_past"] if c in q_feat.columns]
NODE_AGG_COLS_RECV   = [c for c in ["r_cnt_1h_past","r_sum_recv_1h_past","r_mean_recv_1h_past","r_max_recv_1h_past","r_std_recv_1h_past","r_sum_paid_1h_past","r_mean_paid_1h_past","r_cnt_24h_past","r_net_1h_past"] if c in q_feat.columns]

# sender 노드 특성: sender_node 기준으로 평균(또는 max) 집계
q_sender_node_x = (
    q_feat.group_by(["sender_node","bucket_ts"])
          .agg([pl.mean(c).alias(f"nx_{c}") for c in NODE_AGG_COLS_SENDER])
          .rename({"sender_node":"node_key"})
)

# receiver 노드 특성: receiver_node 기준 집계
q_receiver_node_x = (
    q_feat.group_by(["receiver_node","bucket_ts"])
          .agg([pl.mean(c).alias(f"nx_{c}") for c in NODE_AGG_COLS_RECV])
          .rename({"receiver_node":"node_key"})
)

# 두 테이블 합치기: 같은 node_key에 sender/receiver 기반 피처를 모두 채움
q_node_x = (
    q_sender_node_x.join(q_receiver_node_x, on=["node_key","bucket_ts"], how="outer")
)

# q_nodes에 붙이기
q_nodes = (
    q_nodes.join(q_node_x, left_on=["node_key","bucket_ts"], right_on=["node_key","bucket_ts"], how="left")
)

# null -> 0, float32 캐스팅
node_x_cols = [c for c in q_nodes.columns if c.startswith("nx_")]
q_nodes = q_nodes.with_columns([pl.col(c).fill_null(0).cast(pl.Float32) for c in node_x_cols])

print("node feature cols:", node_x_cols[:20], " ... total:", len(node_x_cols))

In [ ]:
# q_feat.select([
#     "ts", "From Account", "To Account",
#     "s_cnt_1h_past", "s_sum_paid_1h_past", # "s_nuniq_to_24h_past",
#     "r_cnt_1h_past", "r_sum_recv_1h_past" # "r_nuniq_from_24h_past"
# ]).head(10).collect()

In [ ]:
# q_feat.select(pl.col("ts").is_null().sum()).collect()

In [ ]:
# q_feat.select(pl.col("ts").dtype).collect()

In [ ]:
# q_feat.schema["ts"]

# 9) 모델에 안 넣을 컬럼 drop    

In [ ]:
q_feat = q_feat.drop(EDGE_ATTR_COLS)

In [ ]:
q_feat = q_feat.drop(NODE_AGG_COLS_SENDER)

In [ ]:
q_feat = q_feat.drop(NODE_AGG_COLS_RECV)

In [ ]:
q_feat = q_feat.drop(EDGE_CAT_COLS)

In [ ]:
DROP_COLS_V1 = [
    # raw time / ids
    "Timestamp", "bucket_ts_str",
    # "ts", "bucket_ts",
    "hour", "day_of_week",

    # account / bank / entity identifiers
    # "From Account", "To Account",
    # "From Bank", "To Bank",
    # "Sender_Bank_Name", "Receiver_Bank_Name",
    # "Sender_Entity", "Receiver_Entity", # 얘를 드랍해야 하는 거 아니던가...

    # intermediate entity cols
    "sender_entity_type", "receiver_entity_type", # 얘를 왜 드랍 하더라,.,,,

    # raw amount columns
    "Amount Paid", "Amount Received",
    "Amount_Paid_USD", "Amount_Received_USD",

    # currency
    # "Receiving Currency", "Payment Currency"

    "sender_node", "receiver_node",

    "_one",
]

q_model = q_feat.drop([c for c in DROP_COLS_V1 if c in q_feat.columns])

In [ ]:
# drop 전 51
print(len(q_model.columns))
print(q_model.columns)

In [ ]:
# FLOAT64_COLS = [
#     "log_amount_paid_usd",
#     "log_amount_received_usd",
#     "risk_x_log_paid",
#     "ach_x_log_paid",
#     "btc_x_log_paid",
# ]

# q_model = q_model.with_columns([
#     pl.col(c).cast(pl.Float32) for c in FLOAT64_COLS
# ])

In [ ]:
q_model.schema

In [ ]:
#“이 컬럼이 numeric / categorical인지” 빠르게 나누기
schema = q_model.schema

numeric_cols = [
    c for c, d in schema.items()
    if d in (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.Float32, pl.Float64,
        pl.Boolean,   # ✅ 추가
    )
]

categorical_cols = [
    c for c, d in schema.items()
    if d in (pl.Utf8, pl.Categorical)
]

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

In [ ]:
#“이 컬럼이 numeric / categorical인지” 빠르게 나누기
schema = q_edges.schema

numeric_cols = [
    c for c, d in schema.items()
    if d in (
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.Float32, pl.Float64,
        pl.Boolean,   # ✅ 추가
    )
]

categorical_cols = [
    c for c, d in schema.items()
    if d in (pl.Utf8, pl.Categorical)
]

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

In [ ]:
def audit_feature_routing(
    q_feat: pl.LazyFrame,
    q_edges: pl.LazyFrame,
    q_nodes: pl.LazyFrame,
    EDGE_ATTR_COLS: list,
    EDGE_CAT_COLS: list,
    node_x_cols: list,
    DROP_COLS_V1: list,
    # 필요하면 너가 직접 META_COLS를 정의해서 넘겨도 됨
    META_COLS: list = None,
    label_col: str = "Is Laundering",
    split_col: str = "split",
    verbose_show: int = 60,
):
    """
    목적:
      - 최종적으로 각 컬럼이 어디로 갔는지(EDGE_ATTR / EDGE_CAT / NODE_X / DROP / META / LEFTOVER) 자동 점검
      - 빠뜨린 피처가 있는지, 중복 라우팅(동시에 drop+사용 등) 있는지 잡아내기

    전제:
      - q_feat: edge row 단위 테이블 (sender/receiver node 붙기 전/후 상관없지만 보통 q_feat)
      - q_edges: edge row 단위 테이블 (src_id/dst_id 붙은 버전)
      - q_nodes: node 단위 테이블 (node_x_cols 포함)
    """

    META_COLS = META_COLS or [
        # 트랜잭션 메타(운영 KPI/리포팅에 보통 필요한 것들)
        "ts", "bucket_ts", "ts_day",
        "From Account", "To Account",
        "src_id", "dst_id",
        split_col, "node_split",
        label_col,
        "node_id", "account_id", "y",
    ]

    # LazyFrame 컬럼 목록
    feat_cols = set(q_feat.columns)
    edge_cols = set(q_edges.columns)
    node_cols = set(q_nodes.columns)

    # 라우팅 대상 sets
    s_edge_attr = set(EDGE_ATTR_COLS)
    s_edge_cat  = set(EDGE_CAT_COLS)
    s_node_x    = set(node_x_cols)
    s_drop      = set([c for c in DROP_COLS_V1 if c in feat_cols])
    s_meta      = set([c for c in META_COLS if (c in feat_cols) or (c in edge_cols) or (c in node_cols)])

    # 1) 오버랩(충돌) 체크
    def inter(a, b): return sorted(list(a & b))
    overlaps = {
        "EDGE_ATTR ∩ EDGE_CAT": inter(s_edge_attr, s_edge_cat),
        "EDGE_ATTR ∩ DROP": inter(s_edge_attr, s_drop),
        "EDGE_CAT ∩ DROP": inter(s_edge_cat, s_drop),
        "NODE_X ∩ DROP": inter(s_node_x, s_drop),
        "NODE_X ∩ META": inter(s_node_x, s_meta),
        "EDGE_ATTR ∩ META": inter(s_edge_attr, s_meta),
        "EDGE_CAT ∩ META": inter(s_edge_cat, s_meta),
    }

    # 2) LEFTOVER 정의:
    #    - q_feat에 남아 있는데 EDGE_ATTR로도, EDGE_CAT로도, DROP로도, META로도 지정되지 않은 컬럼
    routed_from_feat = (s_edge_attr | s_edge_cat | s_drop | (s_meta & feat_cols))
    leftover_feat = sorted(list(feat_cols - routed_from_feat))

    # 3) q_edges/q_nodes 쪽도 체크(의도치 않은 컬럼이 엄청 남아있는지)
    #    - edges는 src_id/dst_id/split + EDGE_ATTR + EDGE_CAT_id 정도만 남는게 보통
    expected_edges = set(["src_id", "dst_id", split_col]) | s_edge_attr | set([f"{c}__id" for c in s_edge_cat if c]) | set(META_COLS)
    extra_edges = sorted(list(edge_cols - expected_edges))

    #    - nodes는 node_id/account_id/bucket_ts/node_split/y + node_x_cols 정도만 남는게 보통
    expected_nodes = set(["node_id", "account_id", "bucket_ts", "node_split", "y", "node_key"]) | s_node_x | set(META_COLS)
    extra_nodes = sorted(list(node_cols - expected_nodes))

    # 4) 출력
    print("\n" + "="*80)
    print("[Feature Routing Audit]")
    print("- q_feat cols :", len(feat_cols))
    print("- q_edges cols:", len(edge_cols))
    print("- q_nodes cols:", len(node_cols))
    print("- EDGE_ATTR_COLS:", len(s_edge_attr))
    print("- EDGE_CAT_COLS :", len(s_edge_cat))
    print("- NODE_X cols   :", len(s_node_x))
    print("- DROP in q_feat:", len(s_drop))
    print("- META (present):", len(s_meta))
    print("="*80)

    print("\n[Overlaps / Conflicts] (비어있어야 이상적)")
    any_bad = False
    for k, v in overlaps.items():
        if len(v) > 0:
            any_bad = True
            print(f"- {k} ({len(v)}): {v[:verbose_show]}")
    if not any_bad:
        print("- OK (no overlaps)")

    print("\n[LEFTOVER in q_feat] (여기에 많으면 '빠뜨린 피처'일 확률 큼)")
    if len(leftover_feat) == 0:
        print("- OK (no leftover)")
    else:
        print(f"- leftover ({len(leftover_feat)}): {leftover_feat[:verbose_show]}")

    print("\n[Extra columns in q_edges] (의도치 않게 메타/원본이 너무 남아있나 체크)")
    if len(extra_edges) == 0:
        print("- OK (no extra edges cols)")
    else:
        print(f"- extra_edges ({len(extra_edges)}): {extra_edges[:verbose_show]}")

    print("\n[Extra columns in q_nodes]")
    if len(extra_nodes) == 0:
        print("- OK (no extra nodes cols)")
    else:
        print(f"- extra_nodes ({len(extra_nodes)}): {extra_nodes[:verbose_show]}")

    print("\n[Quick sanity: required columns 존재 여부]")
    required_feat = [label_col, split_col, "From Account", "To Account"]
    missing_req = [c for c in required_feat if c not in feat_cols]
    print("- missing in q_feat:", missing_req if missing_req else "OK")

    required_edges = ["src_id", "dst_id", split_col]
    missing_e = [c for c in required_edges if c not in edge_cols]
    print("- missing in q_edges:", missing_e if missing_e else "OK")

    required_nodes = ["node_id", "account_id", "bucket_ts", "node_split", "y"]
    missing_n = [c for c in required_nodes if c not in node_cols]
    print("- missing in q_nodes:", missing_n if missing_n else "OK")

    print("="*80 + "\n")

In [ ]:
# Step 9 끝 ~ Step 10 시작 직전 어딘가에:
EDGE_CAT_COLS = [
    "dow_cat",
    "timeofday_bucket",
    "Payment Currency",
    "Receiving Currency",
    "Sender_Bank_Name",
    "Receiver_Bank_Name",
    "sender_entity_type",
    "receiver_entity_type",
    "From Bank",
    "To Bank",
]  # 아직 안 쓰면 빈 리스트라도 OK
# META_COLS는 필요하면 너가 커스텀해서 넣어도 됨

audit_feature_routing(
    q_feat=q_feat,
    q_edges=q_edges,
    q_nodes=q_nodes,
    EDGE_ATTR_COLS=EDGE_ATTR_COLS,
    EDGE_CAT_COLS=EDGE_CAT_COLS,
    node_x_cols=node_x_cols,
    DROP_COLS_V1=DROP_COLS_V1,
    META_COLS=None,   # 기본값 사용(위 함수 안의 기본 META)
    label_col="Is Laundering",
    split_col="split",
    verbose_show=80
)

# 10) 마지막에 pandas로 collect

In [ ]:
# =========================================================
# (10-GNN) Collect -> torch tensors -> PyG Data
# =========================================================

# ---- (1) Node table collect
nodes_df = (
    q_nodes
    .select(["node_id", "account_id", "bucket_ts", "node_split", "y"] + node_x_cols)
    .collect()
    .to_pandas()
)

# node features
x = torch.tensor(nodes_df[node_x_cols].values, dtype=torch.float32)

# label (BCE용 float 권장)
# (선택) BCEWithLogitsLoss 쓸 거면 shape [N] 또는 [N,1] 둘 다 가능.
y = torch.tensor(nodes_df["y"].values, dtype=torch.float32)  # shape [N]

# node split masks
train_mask = torch.tensor(nodes_df["node_split"].values == "train", dtype=torch.bool)
val_mask   = torch.tensor(nodes_df["node_split"].values == "val",   dtype=torch.bool)
test_mask  = torch.tensor(nodes_df["node_split"].values == "test",  dtype=torch.bool)

# train 기준 mean/std
x_train = x[train_mask]
mu = x_train.mean(dim=0, keepdim=True)
std = x_train.std(dim=0, keepdim=True).clamp_min(1e-6)

x = (x - mu) / std

EDGE_CAT_ID_COLS = [f"{c}__id" for c in EDGE_CAT_COLS]

select_cols = ["src_id", "dst_id", "split"] + EDGE_ATTR_COLS
if "EDGE_CAT_ID_COLS" in globals() and len(EDGE_CAT_ID_COLS) > 0:
    select_cols = select_cols + EDGE_CAT_ID_COLS

edges_df = (
    q_edges
    .select(select_cols)
    .collect()
    .to_pandas()
)

edge_index_fwd = torch.tensor(edges_df[["src_id", "dst_id"]].values.T, dtype=torch.long)
edge_index_rev = torch.tensor(edges_df[["dst_id", "src_id"]].values.T, dtype=torch.long)
edge_index = torch.cat([edge_index_fwd, edge_index_rev], dim=1)

# edge_attr: numeric only (EDGE_ATTR_COLS가 비면 None)
edge_attr = (
    torch.tensor(edges_df[EDGE_ATTR_COLS].values, dtype=torch.float32)
    if len(EDGE_ATTR_COLS) > 0
    else None
)

# ✅ edge_cat: categorical ids
edge_cat = None
edge_cat_sizes = {}
if "EDGE_CAT_ID_COLS" in globals() and len(EDGE_CAT_ID_COLS) > 0:
    edge_cat = torch.tensor(edges_df[EDGE_CAT_ID_COLS].values, dtype=torch.long)
    edge_cat_sizes = cat_sizes  # Step 8-1-B에서 만든 dict

# ---- (3) PyG Data
data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_attr,
    y=y
)

data.x = x
data.x_mu = mu
data.x_std = std

# ✅ attach edge_cat + sizes
data.edge_cat = edge_cat
data.edge_cat_sizes = edge_cat_sizes

# ---- (4) masks 저장 (node masks)
data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

# ---- (5) edge split
edge_split = edges_df["split"].values  # numpy array of strings

# ✅ 메시지패싱용(엄격): train만 사용
data.edge_mp_train = torch.tensor(edge_split == "train", dtype=torch.bool)
data.edge_mp_val   = torch.tensor(edge_split != "test", dtype=torch.bool)   # ✅ 너가 말한 "진짜 엄격"
data.edge_mp_test  = torch.tensor(edge_split == edge_split,  dtype=torch.bool)   # ✅ test도 train+val까지만 (선택C)

# ✅ KPI/평가용(거래 자체 평가)
data.edge_loss_train = torch.tensor(edge_split == "train", dtype=torch.bool)
data.edge_loss_val   = torch.tensor(edge_split == "val",   dtype=torch.bool)
data.edge_loss_test  = torch.tensor(edge_split == "test",  dtype=torch.bool)

# edge_attr normalize (train edges 기준)
if data.edge_attr is not None:
    e_train = data.edge_attr[data.edge_mp_train]
    emu = e_train.mean(dim=0, keepdim=True)
    estd = e_train.std(dim=0, keepdim=True).clamp_min(1e-6)
    data.edge_attr = (data.edge_attr - emu) / estd

# ---- (6) asserts (Data 만든 직후, masks까지 붙인 뒤가 가장 좋음)
assert (data.edge_attr is None) or (data.edge_attr.size(0) == data.edge_index.size(1)), \
    "edge_attr row count must match num_edges"

assert int(data.edge_index.max()) < data.num_nodes, \
    "edge_index has node id >= num_nodes (src/dst mapping 확인 필요)"

assert data.x.size(0) == data.num_nodes, \
    "x row count must match num_nodes"

# ---- (7) debug prints
print(data)
print(
    "x:", data.x.shape,
    "y:", data.y.shape,
    "edge_index:", data.edge_index.shape,
    "edge_attr:", (None if data.edge_attr is None else data.edge_attr.shape)
)
print("train/val/test nodes:", int(data.train_mask.sum()), int(data.val_mask.sum()), int(data.test_mask.sum()))
print("train/val/test mp edges:", int(data.edge_mp_train.sum()), int(data.edge_mp_val.sum()), int(data.edge_mp_test.sum()))
print("train/val/test loss edges:", int(data.edge_loss_train.sum()), int(data.edge_loss_val.sum()), int(data.edge_loss_test.sum()))

In [ ]:
# ---- (2-0) META DF들 만들기 (평가/KPI용)
EDGE_META_COLS = [
    "src_id", "dst_id", "split",
    "Is Laundering",
    "ts_day",
    "From Account", "To Account",
    "bucket_ts",
    "Pattern_Type", "Pattern_Detail",
]
EDGE_META_COLS = [c for c in EDGE_META_COLS if c in q_edges.columns]

edges_df_meta = (
    q_edges
    .select(EDGE_META_COLS)
    .collect()
    .to_pandas()
)

NODE_META_COLS = ["node_id", "account_id", "bucket_ts", "node_split", "y"]
nodes_df_meta = (
    q_nodes
    .select(NODE_META_COLS)
    .collect()
    .to_pandas()
)

# 11) Top-k 로직(하루 기준/누적 기준)

In [ ]:
# Top-k 로직 교체용 함수:“k개 적발하려면 몇 건을 봐야 하는지”를 계산하는 함수
def workload_to_find_k_positives(y_true, y_score, k_pos):
    """
    목표: true positive를 k_pos개 '찾기' 위해
         상위 점수부터 몇 건(N)을 조사해야 하는지 계산

    Returns:
      N_required: 필요한 조사 건수
      precision:  k_found / N_required
      recall:     k_found / total_pos
      f1:         f1 (top-N을 양성으로 가정한 경우)
      k_found:    실제로 찾은 TP 수 (total_pos < k_pos면 total_pos)
    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)

    total_pos = y_true.sum()
    if total_pos == 0:
        return 0, 0.0, 0.0, 0.0, 0

    # 점수 내림차순 정렬
    order = np.argsort(-y_score)
    y_sorted = y_true[order]

    # 누적 TP
    cum_tp = np.cumsum(y_sorted)

    target = min(k_pos, int(total_pos))

    # cum_tp >= target 되는 첫 index
    idx = np.searchsorted(cum_tp, target, side="left")
    N_required = int(idx + 1)  # index -> count

    k_found = int(cum_tp[idx])  # 보통 target과 같음(동점/중복 없으면)

    precision = k_found / (N_required + 1e-12)
    recall = k_found / (total_pos + 1e-12)

    # top-N을 양성으로 보면: TP=k_found, FP=N-TP, FN=total_pos-TP
    fp = N_required - k_found
    fn = total_pos - k_found
    f1 = (2 * k_found) / (2 * k_found + fp + fn + 1e-12)

    return N_required, precision, recall, f1, k_found

In [ ]:
def per_day_workload_summary(df, day_col, label_col, score_col, k_pos_list=(30,50,100,200)):
    """
    df: pandas DataFrame with [day_col, label_col, score_col]
    day별로 'k_pos개 적발하기 위해 필요한 조사량 N' 계산 후
    k_pos마다 daily 분포 + mean/median/p90 요약을 반환
    """
    out_rows = []
    for day, g in df.groupby(day_col):
        y = g[label_col].astype(int).values
        s = g[score_col].values
        total_pos = int(y.sum())
        n = len(g)

        for k_pos in k_pos_list:
            N_req, p, r, f1, found = workload_to_find_k_positives(y, s, k_pos)
            out_rows.append({
                "day": day,
                "n_rows": n,
                "pos": total_pos,
                "k_pos": k_pos,
                "N_required": N_req,
                "precision_at_Nk": p,
                "recall_at_Nk": r,
                "f1_at_Nk": f1,
                "found_tp": found,
                "target_k": min(k_pos, total_pos),
                "hit_target": (found >= min(k_pos, total_pos)) and (total_pos > 0)
            })

    daily = pd.DataFrame(out_rows)

    # day에 pos=0이면 N_required=0이 되고 의미가 약하니,
    # 운영 KPI로는 "pos>0인 day"만 요약하는게 보통 더 깔끔함
    daily_pos = daily[daily["pos"] > 0].copy()

    def summarize(sub):
        # N_required는 "작을수록 좋음"
        return pd.Series({
            "days": sub["day"].nunique(),
            "mean_N_required": sub["N_required"].mean(),
            "median_N_required": sub["N_required"].median(),
            "p90_N_required": sub["N_required"].quantile(0.90),
            "mean_precision": sub["precision_at_Nk"].mean(),
            "median_precision": sub["precision_at_Nk"].median(),
            "p90_precision": sub["precision_at_Nk"].quantile(0.90),
        })

    summary = (
        daily_pos
        .groupby("k_pos", as_index=False)
        .apply(summarize)
        .reset_index(drop=True)
    )

    return daily, summary

In [ ]:
def log_per_day_kpi_to_wandb(split_name, df_raw_with_day, y_true, y_score, k_list=(30,50,100,200)):
    """
    split_name: "val" or "test"
    df_raw_with_day: pandas DF that includes column "ts_day" aligned with y_true/y_score index
    y_true/y_score: numpy-like aligned arrays
    """
    tmp = pd.DataFrame({
        "ts_day": df_raw_with_day["ts_day"].values,
        "label": np.asarray(y_true).astype(int),
        "score": np.asarray(y_score),
    })

    daily, summary = per_day_workload_summary(
        tmp,
        day_col="ts_day",
        label_col="label",
        score_col="score",
        k_pos_list=k_list
    )

    # (A) 요약 로그: k별 mean/median/p90
    for _, row in summary.iterrows():
        k = int(row["k_pos"])
        wandb.log({
            f"{split_name}_perday_find_{k}_mean_N": float(row["mean_N_required"]),
            f"{split_name}_perday_find_{k}_median_N": float(row["median_N_required"]),
            f"{split_name}_perday_find_{k}_p90_N": float(row["p90_N_required"]),
            f"{split_name}_perday_find_{k}_mean_precision": float(row["mean_precision"]),
            f"{split_name}_perday_find_{k}_median_precision": float(row["median_precision"]),
            f"{split_name}_perday_find_{k}_p90_precision": float(row["p90_precision"]),
            f"{split_name}_perday_days_with_pos_{k}": int(row["days"]),
        })

    # (B) 분포 확인용 테이블(너무 크면 상위 일부만)
    # day*k 만큼 행이 생김 -> 기간 짧으면 OK, 길면 샘플링/요약만 남기기
    table = wandb.Table(dataframe=daily.head(5000))
    wandb.log({f"{split_name}_perday_workload_table": table})

In [ ]:
def topN_distinct_account_metrics(df_meta, y_true, y_score, topN,
                                  account_cols=("From Account","To Account"),
                                  score_agg="max"):
    """
    전체 기간에서 score 내림차순 topN '거래'를 뽑고,
    그 안에서 account_cols(From/To) 기준 distinct 계좌로 묶어 평가.

    label: 계좌가 topN 거래 중 laundering 거래가 1번이라도 있으면 1 (max)
    score: 계좌 내 score 집계 (max 추천)
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    tmp = df_meta[list(account_cols)].copy()
    tmp["label_trx"] = y
    tmp["score_trx"] = s

    # topN 거래 선택
    tmp = tmp.sort_values("score_trx", ascending=False).head(int(min(topN, len(tmp))))

    # 거래 -> 계좌 집계
    agg = aggregate_to_account_level(
        df_meta=tmp, y_true=tmp["label_trx"].values, y_score=tmp["score_trx"].values,
        account_cols=account_cols, score_agg=score_agg
    )

    y_acc = agg["label"].values
    tp = int((y_acc == 1).sum())
    fp = int((y_acc == 0).sum())
    precision = tp / (tp + fp + 1e-12)

    return {
        "topN_trx": int(min(topN, len(tmp))),
        "distinct_accounts": int(len(agg)),
        "tp_accounts": tp,
        "fp_accounts": fp,
        "precision_accounts": float(precision),
    }

In [ ]:
def score_bin_distribution(y_true, y_score, n_bins=10, score_scale=1000):
    """
    score를 0~score_scale로 스케일링한 뒤,
    bin별 label(0/1) 건수 집계.
    항상 normal_cnt / laundering_cnt 컬럼이 존재하도록 보정.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    s_scaled = np.clip(s * score_scale, 0, score_scale)
    bins = np.linspace(0, score_scale, n_bins + 1)

    # 0..n_bins-1
    bin_idx = np.digitize(s_scaled, bins, right=False) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)

    df = pd.DataFrame({"bin": bin_idx, "label": y})

    # label별 count (항상 0/1 둘 다 컬럼을 만들도록 reindex)
    cnt = (
        df.groupby(["bin", "label"])
          .size()
          .unstack(fill_value=0)
          .reindex(columns=[0, 1], fill_value=0)   # ✅ 핵심
          .rename(columns={0: "normal_cnt", 1: "laundering_cnt"})
          .reset_index()
    )

    # bin_range 붙이기
    cnt["bin_range"] = cnt["bin"].apply(lambda i: f"{int(bins[i])}-{int(bins[i+1])}")

    # 보기 좋게 정렬
    cnt = cnt[["bin", "bin_range", "normal_cnt", "laundering_cnt"]]
    return cnt

In [ ]:
def aggregate_to_account_level(df_meta, y_true, y_score,
                               account_cols=("From Account",),
                               score_agg="max"):
    """
    거래 단위 (y_true, y_score)를 계좌 단위로 집계.
    account_cols: ("From Account",) or ("To Account",) or ("From Account","To Account")
    score_agg: "max" (추천) or "mean"
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    # (A) 거래 메타 + label/score 결합
    tmp = df_meta[list(account_cols)].copy()
    tmp["label_trx"] = y
    tmp["score_trx"] = s

    # (B) From/To 둘 다 쓰는 경우: long 형태로 펼치기
    if len(account_cols) == 2:
        a1, a2 = account_cols
        tmp1 = tmp[[a1, "label_trx", "score_trx"]].rename(columns={a1: "account"})
        tmp2 = tmp[[a2, "label_trx", "score_trx"]].rename(columns={a2: "account"})
        tmp_long = pd.concat([tmp1, tmp2], axis=0, ignore_index=True)
    else:
        a1 = account_cols[0]
        tmp_long = tmp[[a1, "label_trx", "score_trx"]].rename(columns={a1: "account"})

    # 결측 계좌 방어
    tmp_long = tmp_long.dropna(subset=["account"])

    # (C) 계좌 단위 집계
    if score_agg == "mean":
        agg = tmp_long.groupby("account").agg(
            label=("label_trx", "max"),
            score=("score_trx", "mean"),
            n_hits=("score_trx", "size"),
        ).reset_index()
    else:  # "max"
        agg = tmp_long.groupby("account").agg(
            label=("label_trx", "max"),
            score=("score_trx", "max"),
            n_hits=("score_trx", "size"),
        ).reset_index()

    return agg  # columns: account, label, score, n_hits

In [ ]:
def log_per_day_account_kpi_to_wandb(split_name, df_meta, y_true, y_score,
                                     k_list=(30,50,100,200),
                                     account_cols=("From Account","To Account"),
                                     score_agg="max"):
    """
    하루마다 (거래->계좌 집계) 후 workload_to_find_k_positives를 계좌단위로 계산.
    """
    y = np.asarray(y_true).astype(int)
    s = np.asarray(y_score)

    tmp = df_meta[["ts_day"] + list(account_cols)].copy()
    tmp["label_trx"] = y
    tmp["score_trx"] = s

    rows = []
    for day, g in tmp.groupby("ts_day"):
        agg = aggregate_to_account_level(
            df_meta=g, y_true=g["label_trx"].values, y_score=g["score_trx"].values,
            account_cols=account_cols, score_agg=score_agg
        )
        y_acc = agg["label"].values
        s_acc = agg["score"].values

        for k_pos in k_list:
            N_req, p, r, f1, found = workload_to_find_k_positives(y_acc, s_acc, k_pos)
            rows.append({
                "ts_day": day,
                "k_pos": k_pos,
                "n_accounts": len(agg),
                "pos_accounts": int(y_acc.sum()),
                "N_required": N_req,
                "precision": p,
                "recall": r,
                "f1": f1,
                "found_tp": found
            })

    daily = pd.DataFrame(rows)
    daily_pos = daily[daily["pos_accounts"] > 0].copy()

    # 요약 로그
    if len(daily_pos) > 0:
        summ = daily_pos.groupby("k_pos").agg(
            days=("ts_day", "nunique"),
            mean_N=("N_required", "mean"),
            median_N=("N_required", "median"),
            p90_N=("N_required", lambda x: x.quantile(0.9)),
            mean_precision=("precision", "mean"),
        ).reset_index()

        for _, r in summ.iterrows():
            k = int(r["k_pos"])
            wandb.log({
                f"{split_name}_acc_perday_find_{k}_mean_N": float(r["mean_N"]),
                f"{split_name}_acc_perday_find_{k}_median_N": float(r["median_N"]),
                f"{split_name}_acc_perday_find_{k}_p90_N": float(r["p90_N"]),
                f"{split_name}_acc_perday_find_{k}_mean_precision": float(r["mean_precision"]),
                f"{split_name}_acc_perday_days_with_pos_{k}": int(r["days"]),
            })

    wandb.log({f"{split_name}_acc_perday_table": wandb.Table(dataframe=daily.head(5000))})

In [ ]:
def make_trx_scores_from_node_probs(edges_meta_df: pd.DataFrame,
                                   node_prob: np.ndarray,
                                   src_col="src_id", dst_col="dst_id",
                                   agg="max") -> np.ndarray:
    """
    edges_meta_df에 src_id/dst_id가 있다고 가정.
    node_prob: shape [num_nodes] (numpy)
    agg: "max" | "mean"
    """
    src = edges_meta_df[src_col].to_numpy().astype(int)
    dst = edges_meta_df[dst_col].to_numpy().astype(int)

    ps = node_prob[src]
    pd_ = node_prob[dst]

    if agg == "mean":
        return 0.5 * (ps + pd_)
    return np.maximum(ps, pd_)

In [ ]:
def get_pattern_analysis_tables(df, prob, thr):
    """
    Pandas 기반으로 패턴별/상세별 Recall을 계산하여 WandB Table 2개를 반환
    """

    # 1. 실제 세탁 거래(1)만 추출하여 분석용 DF 생성
    analysis = pd.DataFrame({
        "Pattern_Type": df["Pattern_Type"],
        "Pattern_Detail": df["Pattern_Detail"],
        "y_true": df["Is Laundering"],
        "is_detected": (prob >= thr).astype(int)
    })
    pattern_df = analysis[analysis["y_true"] == 1].copy()

    if pattern_df.empty:
        return wandb.Table(columns=["Pattern_Type", "Recall_Pct"]), wandb.Table(columns=["Pattern_Type", "Pattern_Detail", "Recall_Pct"])

    # 2. 패턴별 요약 (Summary)
    summary = pattern_df.groupby("Pattern_Type").agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()
    summary["Recall_Pct"] = (summary["Detected_Count"] / summary["Total_Actual"] * 100).round(2)
    summary = summary.sort_values("Recall_Pct") # 잘 못잡는 순서 정렬

    # 3. 상세 영향도 (Detail)
    detail = pattern_df[pattern_df["Pattern_Detail"] != ""].groupby(["Pattern_Type", "Pattern_Detail"]).agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()
    if not detail.empty:
        detail["Recall_Pct"] = (detail["Detected_Count"] / detail["Total_Actual"] * 100).round(2)
        detail = detail.sort_values(["Pattern_Type", "Recall_Pct"])

    return wandb.Table(dataframe=summary), wandb.Table(dataframe=detail)

In [ ]:
def get_pattern_analysis_tables_find_k(df, prob, k_pos_target):
    """
    Find-K 로직 기반으로 패턴별/상세별 Recall을 계산하여 WandB Table 2개를 반환.
    thr 대신 k_pos_target(예: 3000)을 사용하여 상위 N개 조사 범위를 결정.
    """
    y_true = df["Is Laundering"].astype(int).values
    y_score = np.asarray(prob)

    # 1. 외부 함수를 사용하여 k_pos_target개를 찾기 위한 N_required 계산
    # (이미 정의하신 workload_to_find_k_positives 함수를 활용합니다)
    N_req, _, _, _, _ = workload_to_find_k_positives(y_true, y_score, k_pos_target)

    # 2. 분석용 데이터프레임 생성
    analysis = pd.DataFrame({
        "Pattern_Type": df["Pattern_Type"],
        "Pattern_Detail": df["Pattern_Detail"],
        "y_true": y_true,
        "score": y_score
    })

    # 점수 순 정렬 후 상위 N_req개만 'is_detected'로 표시
    analysis = analysis.sort_values("score", ascending=False).reset_index(drop=True)
    analysis["is_detected"] = 0
    analysis.loc[:N_req-1, "is_detected"] = 1

    # 3. 실제 세탁 거래(1)만 추출하여 패턴 분석 진행
    pattern_df = analysis[analysis["y_true"] == 1].copy()

    if pattern_df.empty:
        return (wandb.Table(columns=["Pattern_Type", "Total_Actual", "Detected_Count", "Recall_Pct"]),
                wandb.Table(columns=["Pattern_Type", "Pattern_Detail", "Total_Actual", "Detected_Count", "Recall_Pct"]))

    # 4. 패턴별 요약 (Summary)
    summary = pattern_df.groupby("Pattern_Type").agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()
    summary["Recall_Pct"] = (summary["Detected_Count"] / summary["Total_Actual"] * 100).round(2)
    summary = summary.sort_values("Recall_Pct") # 검출률 낮은 순 정렬

    # 5. 상세 영향도 (Detail)
    detail = pattern_df[pattern_df["Pattern_Detail"] != ""].groupby(["Pattern_Type", "Pattern_Detail"]).agg(
        Total_Actual=("is_detected", "count"),
        Detected_Count=("is_detected", "sum")
    ).reset_index()

    if not detail.empty:
        detail["Recall_Pct"] = (detail["Detected_Count"] / detail["Total_Actual"] * 100).round(2)
        detail = detail.sort_values(["Pattern_Type", "Recall_Pct"])

    return wandb.Table(dataframe=summary), wandb.Table(dataframe=detail)

In [ ]:
def log_detailed_error_analysis(split_name, df, y_true, y_score, threshold, top_features):
    """
    정탐(TP), 오탐(FP), 미탐(FN) 그룹별로 피처 분포와 일자별 추이를 분석하여 WandB에 로깅
    """
    analysis_df = df.copy()
    analysis_df['prob'] = y_score
    analysis_df['pred'] = (y_score >= threshold).astype(int)
    analysis_df['label'] = y_true.astype(int)

    # TP, FP, FN, TN 분류
    def categorize(row):
        if row['label'] == 1 and row['pred'] == 1: return 'TP (정탐)'
        if row['label'] == 0 and row['pred'] == 1: return 'FP (오탐)'
        if row['label'] == 1 and row['pred'] == 0: return 'FN (미탐)'
        return 'TN'

    analysis_df['error_cat'] = analysis_df.apply(categorize, axis=1)

    # (1) 일자별 에러 분포 (정탐/오탐/미탐 건수 추이)
    # 2022/09/14 ~ 2022/09/28 범위의 트렌드 파악용
    error_by_day = analysis_df.groupby(['ts_day', 'error_cat']).size().unstack(fill_value=0).reset_index()

    thr_tag = f"thr{threshold:.2f}"

    wandb.log({f"{split_name}_{thr_tag}_daily_error_trend": wandb.Table(dataframe=error_by_day)})

    # (2) 피처별 에러 특성 분석 (어떤 피처가 오탐/미탐에 영향을 주는가?)
    # 중요도 상위 피처(top_features)에 대해 각 그룹별 평균 또는 빈도 계산
    feat_stats = []
    for cat in ['TP (정탐)', 'FP (오탐)', 'FN (미탐)']:
        sub = analysis_df[analysis_df['error_cat'] == cat]
        if sub.empty: continue

        stat_row = {'category': cat, 'count': len(sub)}
        for f in top_features:
            if f not in sub.columns: continue
            val = sub[f]
            # 수치형 피처: 평균값 비교
            if pd.api.types.is_numeric_dtype(val):
                stat_row[f"{f}_mean"] = round(val.mean(), 4)
            # 범주형 피처: 최빈값(Mode) 비교
            else:
                stat_row[f"{f}_top_mode"] = str(val.mode()[0]) if not val.mode().empty else "N/A"
        feat_stats.append(stat_row)

    wandb.log({f"{split_name}_{thr_tag}_feature_characteristics_by_error": wandb.Table(dataframe=pd.DataFrame(feat_stats))})

In [ ]:
def log_pattern_error_analysis(split_name, df, y_true, y_score, threshold):
    """
    미탐(FN)과 오탐(FP) 건들에 대해 일자별 패턴 분포를 집중 분석
    """
    df_err = df.copy()
    df_err['pred'] = (y_score >= threshold).astype(int)
    df_err['label'] = y_true.astype(int)

    # FN (미탐): 실제 세탁인데 놓친 것
    fn_df = df_err[(df_err['label'] == 1) & (df_err['pred'] == 0)]
    # FP (오탐): 정상인데 세탁으로 예측한 것
    fp_df = df_err[(df_err['label'] == 0) & (df_err['pred'] == 1)]

    thr_tag = f"thr{threshold:.2f}"

    # (1) 미탐(FN) 패턴 분석: 어떤 날에 어떤 패턴을 주로 놓쳤나?
    if not fn_df.empty:
        fn_pattern_day = fn_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='missed_count')
        wandb.log({f"{split_name}_{thr_tag}_missed_patterns_by_day": wandb.Table(dataframe=fn_pattern_day)})

        # 상세 패턴 미탐 (Pattern_Detail)
        fn_detail = fn_df.groupby(['Pattern_Type', 'Pattern_Detail']).size().reset_index(name='fn_count')
        wandb.log({f"{split_name}_{thr_tag}_missed_patterns_detail": wandb.Table(dataframe=fn_detail)})

    # (2) 오탐(FP) 패턴 분석: 오탐된 건들은 주로 어떤 패턴(혹은 일반 거래)인가?
    if not fp_df.empty:
        # 정상 거래에 패턴 정보가 태깅되어 있는 경우(Synthetic 데이터 등) 유용함
        fp_pattern = fp_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='false_alarm_count')
        wandb.log({f"{split_name}_{thr_tag}_false_alarm_patterns_by_day": wandb.Table(dataframe=fp_pattern)})

In [ ]:
def log_find_k_error_analysis(split_name, df, y_true, y_score, k_pos_target, top_features):
    """
    find-K 로직과 호환되는 에러 분석:
    실제 Positive를 k_pos_target개 찾기 위해 필요한 상위 N개 구간을 정해 분석함.
    """
    y_true_arr = np.asarray(y_true).astype(int)
    y_score_arr = np.asarray(y_score)

    # 1. 외부함수 활용하여 필요한 조사 건수(N_req) 계산
    N_req, precision, recall, f1, k_found = workload_to_find_k_positives(y_true_arr, y_score_arr, k_pos_target)

    # 2. 분석용 DF 생성
    analysis_df = df.copy()
    analysis_df['prob'] = y_score_arr
    analysis_df['label'] = y_true_arr

    # 점수 내림차순 정렬 후 상위 N_req개에 대해 pred=1 부여
    analysis_df = analysis_df.sort_values("prob", ascending=False).reset_index(drop=True)
    analysis_df['pred'] = 0
    analysis_df.loc[:N_req-1, 'pred'] = 1  # 0부터 N_req건까지 1 부여

    # 3. TP/FP/FN 분류 (N_req 구간 기준)
    # TP: 상위 N개 중 실제 세탁 (k_found개)
    # FP: 상위 N개 중 실제 정상 (N_req - k_found개)
    # FN: 상위 N개 밖에 있는 실제 세탁 (total_pos - k_found개)
    def categorize(row):
        if row['label'] == 1 and row['pred'] == 1: return 'TP (정탐)'
        if row['label'] == 0 and row['pred'] == 1: return 'FP (오탐)'
        if row['label'] == 1 and row['pred'] == 0: return 'FN (미탐)'
        return 'TN'

    analysis_df['error_cat'] = analysis_df.apply(categorize, axis=1)

    # --- [A] 일자별 분석 (Trend) ---
    # find-K를 위해 조사한 N개 내에서의 일자별 정탐/오탐/미탐 분포
    error_by_day = analysis_df.groupby(['ts_day', 'error_cat']).size().unstack(fill_value=0).reset_index()
    wandb.log({f"{split_name}_find_{k_pos_target}_daily_error_trend": wandb.Table(dataframe=error_by_day)})

    # --- [B] 피처별 분석 (Characteristics) ---
    feat_stats = []
    for cat in ['TP (정탐)', 'FP (오탐)', 'FN (미탐)']:
        sub = analysis_df[analysis_df['error_cat'] == cat]
        if sub.empty: continue

        stat_row = {'category': cat, 'count': len(sub)}
        for f in top_features:
            if f not in sub.columns: continue
            val = sub[f]
            if pd.api.types.is_numeric_dtype(val):
                stat_row[f"{f}_mean"] = round(val.mean(), 4)
            else:
                stat_row[f"{f}_top_mode"] = str(val.mode()[0]) if not val.mode().empty else "N/A"
        feat_stats.append(stat_row)

    wandb.log({f"{split_name}_find_{k_pos_target}_feat_stats": wandb.Table(dataframe=pd.DataFrame(feat_stats))})

    # --- [C] 패턴 중심 분석 ---
    # 미탐(FN): 3000개를 잡기 위해 조사해야 하는 우선순위 밖으로 밀려난 패턴들
    fn_df = analysis_df[analysis_df['error_cat'] == 'FN (미탐)']
    if not fn_df.empty:
        fn_pattern = fn_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='fn_count')
        wandb.log({f"{split_name}_find_{k_pos_target}_missed_patterns_by_day": wandb.Table(dataframe=fn_pattern)})

        fn_detail = fn_df.groupby(['Pattern_Type', 'Pattern_Detail']).size().reset_index(name='fn_count')
        wandb.log({f"{split_name}_find_{k_pos_target}_missed_patterns_detail": wandb.Table(dataframe=fn_detail)})

    # 오탐(FP): 3000개를 잡는 과정에서 조사가 불가피했던 정상 패턴들
    fp_df = analysis_df[analysis_df['error_cat'] == 'FP (오탐)']
    if not fp_df.empty:
        fp_pattern = fp_df.groupby(['ts_day', 'Pattern_Type']).size().reset_index(name='fp_count')
        wandb.log({f"{split_name}_find_{k_pos_target}_false_alarm_patterns_by_day": wandb.Table(dataframe=fp_pattern)})

# 12) GATv2Conv, GATv2NodeClassifier class 정의

In [ ]:
class GATv2Conv(MessagePassing):
    _alpha: OptTensor

    def __init__(self, in_channels: int,
                 out_channels: int, heads: int = 1, concat: bool = True,
                 negative_slope: float = 0.2, dropout: float = 0.,
                 add_self_loops: bool = True, bias: bool = True,
                 share_weights: bool = False,
                 edge_dim: Optional[int] = None,   # ✅ NEW
                 **kwargs):
        super(GATv2Conv, self).__init__(node_dim=0, **kwargs)

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        self.negative_slope = negative_slope
        self.dropout = dropout
        self.add_self_loops = add_self_loops
        self.share_weights = share_weights
        self.edge_dim = edge_dim

        self.lin_l = Linear(in_channels, heads * out_channels, bias=bias)
        if share_weights:
            self.lin_r = self.lin_l
        else:
            self.lin_r = Linear(in_channels, heads * out_channels, bias=bias)

        # ✅ NEW: edge encoder (optional)
        if edge_dim is not None:
            self.lin_edge = Linear(edge_dim, heads * out_channels, bias=False)
        else:
            self.lin_edge = None

        self.att = Parameter(torch.Tensor(1, heads, out_channels))

        if bias and concat:
            self.bias = Parameter(torch.Tensor(heads * out_channels))
        elif bias and not concat:
            self.bias = Parameter(torch.Tensor(out_channels))
        else:
            self.register_parameter('bias', None)

        self._alpha = None
        self.reset_parameters()

    def reset_parameters(self):
        glorot(self.lin_l.weight)
        glorot(self.lin_r.weight)
        glorot(self.att)
        if self.lin_edge is not None:
            glorot(self.lin_edge.weight)   # ✅ NEW
        zeros(self.bias)

    def forward(self, x: Union[Tensor, PairTensor], edge_index: Adj,
                edge_attr: OptTensor = None,        # ✅ NEW
                size: Size = None, return_attention_weights: bool = None):

        H, C = self.heads, self.out_channels

        x_l: OptTensor = None
        x_r: OptTensor = None
        if isinstance(x, Tensor):
            assert x.dim() == 2
            x_l = self.lin_l(x).view(-1, H, C)
            if self.share_weights:
                x_r = x_l
            else:
                x_r = self.lin_r(x).view(-1, H, C)
        else:
            x_l, x_r = x[0], x[1]
            assert x[0].dim() == 2
            x_l = self.lin_l(x_l).view(-1, H, C)
            if x_r is not None:
                x_r = self.lin_r(x_r).view(-1, H, C)

        assert x_l is not None
        assert x_r is not None

        # ---- (A) self-loop는 "projection 전에(2D에서)" 처리하는 게 가장 안전
        edge_attr_2d = edge_attr  # 원래 edge_attr: [E, edge_dim] or None

        if self.add_self_loops and isinstance(edge_index, Tensor):
            num_nodes = x_l.size(0)
            if x_r is not None:
                num_nodes = min(num_nodes, x_r.size(0))
            if size is not None:
                num_nodes = min(size[0], size[1])

            # 1) 기존 self-loop 제거 (edge_attr도 같이)
            edge_index, edge_attr_2d = remove_self_loops(edge_index, edge_attr=edge_attr_2d)

            # 2) self-loop 추가
            if edge_attr_2d is None:
                edge_index, _ = add_self_loops(edge_index, num_nodes=num_nodes)
            else:
                # self-loop용 edge feature를 명시적으로 0벡터로 생성해서 붙이기 (안전)
                loop_attr = edge_attr_2d.new_zeros((num_nodes, edge_attr_2d.size(-1)))  # [N, edge_dim]
                edge_index, edge_attr_2d = add_self_loops(
                    edge_index,
                    edge_attr=edge_attr_2d,
                    fill_value=0.0,     # 2D에서는 안정적
                    num_nodes=num_nodes
                )
                # 일부 버전에선 fill_value만으로도 OK지만, 위 loop_attr 방식도 가능
                # (만약 add_self_loops가 fill_value=0.0을 완전히 보장하지 않는다면)
                # edge_attr_2d = torch.cat([edge_attr_2d, loop_attr], dim=0)

        elif self.add_self_loops and isinstance(edge_index, SparseTensor):
            # SparseTensor는 edge_attr 처리 난이도↑ → 최소 버전에서는 self-loop만
            edge_index = set_diag(edge_index)

        # ---- (B) 이제 projection (2D -> 3D) 수행
        edge_emb: OptTensor = None
        if (edge_attr_2d is not None) and (self.lin_edge is not None):
            edge_emb = self.lin_edge(edge_attr_2d).view(-1, H, C)   # [E(+N), H, C]

        # propagate_type: (x: PairTensor, edge_attr: OptTensor)
        out = self.propagate(edge_index, x=(x_l, x_r), edge_attr=edge_emb, size=size)

        alpha = self._alpha
        self._alpha = None

        if self.concat:
            out = out.view(-1, self.heads * self.out_channels)
        else:
            out = out.mean(dim=1)

        if self.bias is not None:
            out += self.bias

        if isinstance(return_attention_weights, bool):
            assert alpha is not None
            if isinstance(edge_index, Tensor):
                return out, (edge_index, alpha)
            elif isinstance(edge_index, SparseTensor):
                return out, edge_index.set_value(alpha, layout='coo')
        else:
            return out

    def message(self, x_j: Tensor, x_i: Tensor,
                edge_attr: OptTensor,      # ✅ NEW
                index: Tensor, ptr: OptTensor,
                size_i: Optional[int]) -> Tensor:

        # ✅ attention input에 edge embedding을 더함
        if edge_attr is None:
            z = x_i + x_j
        else:
            z = x_i + x_j + edge_attr

        z = F.leaky_relu(z, self.negative_slope)
        alpha = (z * self.att).sum(dim=-1)
        alpha = softmax(alpha, index, ptr, size_i)
        self._alpha = alpha
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        # (A) 최소 변경: attention만 edge_attr로 바꿈
        msg = x_j

        # (B) 옵션: message 값에도 edge 정보를 섞고 싶으면 아래로 교체
        # if edge_attr is not None:
        #     msg = x_j + edge_attr

        return msg * alpha.unsqueeze(-1)

    def __repr__(self):
        return '{}({}, {}, heads={})'.format(self.__class__.__name__,
                                             self.in_channels,
                                             self.out_channels, self.heads)

In [ ]:
class GATv2NodeClassifier(nn.Module):
    def __init__(self, in_dim, hidden_dim, edge_dim=None, num_layers=2, heads=4,
                 dropout=0.2, negative_slope=0.2, add_self_loops=True, out_dim=1,
                 edge_cat_sizes: dict = None,
                 edge_cat_emb_dim: int = 8):
        super().__init__()
        self.dropout = dropout

        self.edge_cat_sizes = edge_cat_sizes or {}
        self.edge_cat_cols = list(self.edge_cat_sizes.keys())

        self.edge_cat_emb_dim = int(edge_cat_emb_dim)

        # ✅ edge categorical embedding tables
        # col별로 vocab 다르니 ModuleDict로
        self.edge_cat_emb = nn.ModuleDict()
        for col, vocab in self.edge_cat_sizes.items():
            self.edge_cat_emb[col] = nn.Embedding(int(vocab), self.edge_cat_emb_dim)

        # ✅ 총 edge input dim = edge_dim(num) + edge_cat_emb_dim * num_cat_cols
        self.edge_cat_total_dim = self.edge_cat_emb_dim * len(self.edge_cat_sizes)
        edge_dim_total = (edge_dim or 0) + self.edge_cat_total_dim
        edge_dim_total = edge_dim_total if edge_dim_total > 0 else None

        convs = []
        convs.append(GATv2Conv(
            in_channels=in_dim,
            out_channels=hidden_dim,
            heads=heads,
            concat=True,
            dropout=dropout,
            negative_slope=negative_slope,
            add_self_loops=add_self_loops,
            share_weights=False,
            edge_dim=edge_dim_total,          # ✅ NEW
        ))

        for _ in range(num_layers - 1):
            convs.append(GATv2Conv(
                in_channels=hidden_dim * heads,
                out_channels=hidden_dim,
                heads=heads,
                concat=True,
                dropout=dropout,
                negative_slope=negative_slope,
                add_self_loops=add_self_loops,
                share_weights=False,
                edge_dim=edge_dim_total,      # ✅ NEW
            ))

        self.convs = nn.ModuleList(convs)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * heads, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim)
        )

        self.node_in_norm = nn.LayerNorm(in_dim)
        self.edge_in_norm = nn.LayerNorm(edge_dim_total) if edge_dim_total is not None else None

    def forward(self, x, edge_index, edge_attr=None, edge_cat=None):
        x = self.node_in_norm(x)

        # edge_cat: [E, num_cat_cols] long
        if (edge_cat is not None) and (len(self.edge_cat_sizes) > 0):
            # edge_cat 컬럼 순서가 중요: edges_df에서 만든 순서와 동일해야 함
            embs = []
            cols = self.edge_cat_sizes
            for j, col in enumerate(cols):
                embs.append(self.edge_cat_emb[col](edge_cat[:, j]))  # [E, emb_dim]
            edge_cat_feat = torch.cat(embs, dim=1)  # [E, emb_dim * num_cols]

            if edge_attr is None:
                edge_attr = edge_cat_feat
            else:
                edge_attr = torch.cat([edge_attr, edge_cat_feat], dim=1)

        if edge_attr is not None and self.edge_in_norm is not None:
            edge_attr = self.edge_in_norm(edge_attr)

        for conv in self.convs:
            x = conv(x, edge_index, edge_attr=edge_attr)
            x = F.elu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        logit = self.classifier(x).squeeze(-1)
        return logit

In [ ]:
@torch.no_grad()
def infer_all_node_probs(model, data_mp, device, fanout, batch_size):
    model.eval()
    num_nodes = data_mp.num_nodes
    out = np.zeros(num_nodes, dtype=np.float32)

    loader = NeighborLoader(
        data_mp,
        input_nodes=torch.arange(num_nodes),
        num_neighbors=fanout,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    for batch in loader:
        batch = batch.to(device)
        logit = model(
            batch.x, batch.edge_index,
            getattr(batch, "edge_attr", None),
            edge_cat=getattr(batch, "edge_cat", None)
        )
        bs = batch.batch_size
        prob = torch.sigmoid(logit[:bs]).detach().cpu().numpy()

        n_id = batch.n_id[:bs].detach().cpu().numpy().astype(int)
        out[n_id] = prob

    return out

# 13) train, eval 관련 함수, train_eval_sweep 함수 전 외부에서 정의

In [ ]:
def make_edge_subgraph_keep_nodes(data: Data, edge_mask: torch.Tensor) -> Data:
    d = Data()
    d.x = data.x
    d.y = data.y
    d.edge_index = data.edge_index[:, edge_mask]
    if getattr(data, "edge_attr", None) is not None:
        d.edge_attr = data.edge_attr[edge_mask]
    if getattr(data, "edge_cat", None) is not None and data.edge_cat is not None:
        d.edge_cat = data.edge_cat[edge_mask]

    # masks는 그대로 참조
    d.train_mask = data.train_mask
    d.val_mask   = data.val_mask
    d.test_mask  = data.test_mask

    return d

def safe_auc_metrics(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score)
    if len(np.unique(y_true)) <= 1:
        return np.nan, np.nan
    return roc_auc_score(y_true, y_score), average_precision_score(y_true, y_score)

@torch.no_grad()
def eval_loader(model, loader, device, return_seed_probs=False):
    model.eval()

    ys, ps = [], []
    for batch in loader:
        batch = batch.to(device)

        logit = model(
            batch.x,
            batch.edge_index,
            getattr(batch, "edge_attr", None),
            edge_cat=getattr(batch, "edge_cat", None)
        )

        bs = batch.batch_size  # seed node count
        prob = torch.sigmoid(logit[:bs]).detach().cpu().numpy()
        y = batch.y[:bs].detach().cpu().numpy().astype(int)

        ys.append(y); ps.append(prob)

    y_true = np.concatenate(ys) if len(ys) else np.array([])
    y_prob = np.concatenate(ps) if len(ps) else np.array([])

    auroc, auprc = safe_auc_metrics(y_true, y_prob)
    res = {"auroc": auroc, "auprc": auprc, "y_true": y_true, "y_prob": y_prob}
    if return_seed_probs:
        res["seed_prob_all"] = y_prob  # seed 노드 순서대로 이어붙인 것(아래 KPI에서 안 씀)
    return res

def train_one_run(model, train_loader, val_loader, test_loader,
                  optimizer, criterion, device,
                  max_epochs=50, log_fn=None, early_patience=10):

    best_val = -1.0
    best_state = None
    bad = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        total_seed = 0

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()

            logit = model(
                batch.x, batch.edge_index,
                getattr(batch, "edge_attr", None),
                edge_cat=getattr(batch, "edge_cat", None)
            )

            bs = batch.batch_size
            loss = criterion(logit[:bs], batch.y[:bs].float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += float(loss.item()) * bs
            total_seed += bs

        train_loss = total_loss / max(total_seed, 1)

        val_res  = eval_loader(model, val_loader, device)
        test_res = eval_loader(model, test_loader, device)

        print(
            epoch,
            train_loss,
            val_res["auprc"], val_res["auroc"],
            test_res["auprc"], test_res["auroc"]
        )

        if log_fn is not None:
            log_fn({
                "epoch": epoch,
                "train_loss": float(train_loss),
                "val_auprc": float(val_res["auprc"]) if val_res["auprc"] == val_res["auprc"] else np.nan,
                "val_auroc": float(val_res["auroc"]) if val_res["auroc"] == val_res["auroc"] else np.nan,
                "test_auprc": float(test_res["auprc"]) if test_res["auprc"] == test_res["auprc"] else np.nan,
                "test_auroc": float(test_res["auroc"]) if test_res["auroc"] == test_res["auroc"] else np.nan,
            })

        cur = val_res["auprc"]
        if cur == cur and cur > best_val:
            best_val = cur
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= early_patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # 최종
    val_res  = eval_loader(model, val_loader, device)
    test_res = eval_loader(model, test_loader, device)
    return best_val, val_res, test_res

In [ ]:
def f1_at_threshold(y_true, y_prob, thr):
    y_hat = (y_prob >= thr).astype(int)
    return f1_score(y_true, y_hat), precision_score(y_true, y_hat), recall_score(y_true, y_hat)

def threshold_for_recall(y_true, y_prob, target_recall=0.9):
    # recall_curve: thresholds 길이가 len(precision)-1라서 맞춰 처리
    precision, recall, thr = precision_recall_curve(y_true, y_prob)
    # recall은 high->low가 아니라 케이스마다 달라서 "recall >= target"인 threshold 중 가장 큰 thr를 택하는 방식이 안전
    cand = np.where(recall[:-1] >= target_recall)[0]
    if len(cand) == 0:
        return 1.0  # 달성 불가면 극단값
    return float(thr[cand[-1]])

In [ ]:
# def node_feature_permutation_importance(model, data, node_mask, edge_mask, feature_names,
#                                         device, n_repeats=1, max_feats=30):
#     model.eval()
#     base = eval_loader(model, data, node_mask, device, edge_mask)
#     base_score = base["auprc"]

#     # 너무 많으면 비용 커서 상위 max_feats만
#     feat_idx = list(range(len(feature_names)))[:max_feats]

#     importances = []
#     for j in feat_idx:
#         drops = []
#         for _ in range(n_repeats):
#             d = data.clone().to(device)
#             x = d.x.clone()

#             perm = torch.randperm(x.size(0), device=device)
#             x[:, j] = x[perm, j]
#             d.x = x

#             res = eval_loader(model, d, node_mask, device, edge_mask)
#             drops.append(base_score - res["auprc"])
#         importances.append((feature_names[j], float(np.mean(drops))))

#     imp_df = pd.DataFrame(importances, columns=["feature", "auprc_drop"]).sort_values("auprc_drop", ascending=False)
#     return base_score, imp_df

In [ ]:
def build_loaders(data_mp_train, data_mp_val, data_mp_test, cfg, seed=42):
    seed_everything(seed)

    fanout = list(cfg.fanout)          # e.g. [15, 10]
    batch_size = int(cfg.batch_size)

    train_loader = NeighborLoader(
        data_mp_train,
        input_nodes=data_mp_train.train_mask,   # seed nodes
        num_neighbors=fanout,
        batch_size=batch_size,
        shuffle=True,                 # 재현성 우선
        num_workers=0,                 # ✅ 랜덤성/재현성 안정
    )

    val_loader = NeighborLoader(
        data_mp_val,
        input_nodes=data_mp_val.val_mask,
        num_neighbors=fanout,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    test_loader = NeighborLoader(
        data_mp_test,
        input_nodes=data_mp_test.test_mask,
        num_neighbors=fanout,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )
    return train_loader, val_loader, test_loader

In [ ]:
import tqdm

@torch.no_grad()
def extract_gat_embeddings(model, data_obj, mask, device, out_path, hidden_dim):
    """
    GATv2 노드 분류 모델에서 노드 임베딩을 추출하고 저장
    """
    model.eval()
    # 전체 노드에 대한 임베딩 계산 (메모리 주의: 크면 미니배치로 나눠야 함)
    # 여기서는 간단하게 NeighborLoader 방식으로 추출합니다.

    loader = NeighborLoader(
        data_obj,
        input_nodes=mask,
        num_neighbors=[-1, -1], # 전체 이웃 참조 (추천)
        batch_size=1024,
        shuffle=False
    )

    node_embeddings = []
    node_indices = []

    for batch in tqdm.tqdm(loader, desc="Extracting Nodes"):
        batch = batch.to(device)
        # 1. GNN 레이어 통과 (classifier 직전까지)
        x = model.node_in_norm(batch.x)

        # edge_attr 결합 로직 (forward 복사)
        edge_attr = batch.edge_attr
        if (batch.edge_cat is not None) and (len(model.edge_cat_sizes) > 0):
            embs = []
            for j, col in enumerate(model.edge_cat_cols):
                embs.append(model.edge_cat_emb[col](batch.edge_cat[:, j]))
            edge_cat_feat = torch.cat(embs, dim=1)
            edge_attr = edge_cat_feat if edge_attr is None else torch.cat([edge_attr, edge_cat_feat], dim=1)

        if edge_attr is not None and model.edge_in_norm is not None:
            edge_attr = model.edge_in_norm(edge_attr)

        for conv in model.convs:
            x = conv(x, batch.edge_index, edge_attr=edge_attr)
            x = F.elu(x)

        # 2. Seed 노드(현재 배치의 대상 노드)의 임베딩만 추출
        node_embeddings.append(x[:batch.batch_size].cpu().numpy())
        node_indices.append(batch.n_id[:batch.batch_size].cpu().numpy())

    # 저장 로직 (Parquet)
    final_emb = np.concatenate(node_embeddings)
    final_idx = np.concatenate(node_indices)

    df_emb = pd.DataFrame(final_emb, columns=[f"gnn_node_{i}" for i in range(final_emb.shape[1])])
    df_emb["node_id"] = final_idx
    df_emb.to_parquet(out_path)
    print(f"✅ Saved {out_path}, shape: {df_emb.shape}")

# 14-1) W&B Sweeps용 train/eval 함수 정의(GATv2)

In [ ]:
def train_eval_sweep():
    global data

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cuda":
        torch.cuda.empty_cache()

    run = wandb.init()
    cfg = wandb.config   # ✅ cfg 먼저 확정

    # -------------------------
    # 0) MP 서브그래프(노드 유지 버전)
    # -------------------------
    data_mp_train = make_edge_subgraph_keep_nodes(data, data.edge_mp_train)
    data_mp_val   = make_edge_subgraph_keep_nodes(data, data.edge_mp_val)
    data_mp_test  = make_edge_subgraph_keep_nodes(data, data.edge_mp_test)

    # ✅ loader는 cfg 확보 후 생성해야 함
    train_loader, val_loader, test_loader = build_loaders(
        data_mp_train, data_mp_val, data_mp_test, cfg, seed=42
    )

    d = data  # global reference

    # -------------------------
    # 1) model
    # -------------------------
    edge_dim = d.edge_attr.size(1) if getattr(d, "edge_attr", None) is not None else None
    edge_cat_sizes = getattr(d, "edge_cat_sizes", {})

    model = GATv2NodeClassifier(
        in_dim=d.x.size(1),
        hidden_dim=int(cfg.hidden_dim),
        edge_dim=edge_dim,
        num_layers=int(cfg.num_layers),
        heads=int(cfg.heads),
        dropout=float(cfg.dropout),
        negative_slope=float(cfg.negative_slope),
        add_self_loops=bool(cfg.add_self_loops),
        edge_cat_sizes=edge_cat_sizes,
        edge_cat_emb_dim=int(cfg.edge_cat_emb_dim),
    ).to(device)

    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    # -------------------------
    # 2) loss/opt
    # -------------------------
    pos = int(d.y[d.train_mask].sum().item())
    neg = int(d.train_mask.sum().item()) - pos
    base = (neg / max(pos, 1))
    pos_weight = torch.tensor([base * float(cfg.pos_weight_factor)],
                              device=device, dtype=torch.float)

    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(cfg.lr),
        weight_decay=float(cfg.weight_decay),
    )

    wandb.log({
        "train_pos": pos,
        "train_neg": neg,
        "pos_weight_base": float(base),
        "pos_weight_used": float(pos_weight.item()),
        "num_nodes": int(d.num_nodes),
        "num_edges": int(d.edge_index.size(1)),
    })

    # -------------------------
    # 3) train (mini-batch)
    # -------------------------
    best_val, val_res, test_res = train_one_run(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        max_epochs=int(cfg.max_epochs),
        early_patience=int(cfg.early_patience),
        log_fn=lambda x: wandb.log(x),
    )

    wandb.log({
        "best_val_auprc": float(best_val) if best_val == best_val else np.nan,
        "val_auprc_final": float(val_res["auprc"]) if val_res["auprc"] == val_res["auprc"] else np.nan,
        "val_auroc_final": float(val_res["auroc"]) if val_res["auroc"] == val_res["auroc"] else np.nan,
        "test_auprc_final": float(test_res["auprc"]) if test_res["auprc"] == test_res["auprc"] else np.nan,
        "test_auroc_final": float(test_res["auroc"]) if test_res["auroc"] == test_res["auroc"] else np.nan,
    })

    # -------------------------
    # 4) KPI용: 전체 노드 확률 추론 (반드시 이걸 사용)
    # -------------------------
    fanout = list(cfg.fanout)
    batch_size = int(cfg.batch_size)

    # (추천) inference는 train보다 fanout을 더 보수적으로:
    # fanout_infer = [min(5, fanout[0]), min(5, fanout[1])] if len(fanout) >= 2 else fanout
    fanout_infer = fanout

    val_node_prob_all  = infer_all_node_probs(model, data_mp_val,  device, fanout_infer, batch_size)
    test_node_prob_all = infer_all_node_probs(model, data_mp_test, device, fanout_infer, batch_size)

    # -------------------------
    # 5) KPI (거래 단위)
    # -------------------------
    with torch.no_grad():
        try:
            EDGE_LABEL_COL = "Is Laundering"
            DAY_COL = "ts_day"

            # ✅ 여기 반드시 infer 결과를 사용
            val_node_prob  = val_node_prob_all
            test_node_prob = test_node_prob_all

            val_edges  = edges_df_meta[d.edge_loss_val.cpu().numpy()].copy()
            test_edges = edges_df_meta[d.edge_loss_test.cpu().numpy()].copy()

            val_trx_score  = make_trx_scores_from_node_probs(val_edges,  val_node_prob,  agg="max")
            test_trx_score = make_trx_scores_from_node_probs(test_edges, test_node_prob, agg="max")

            val_trx_y  = val_edges[EDGE_LABEL_COL].astype(int).to_numpy()
            test_trx_y = test_edges[EDGE_LABEL_COL].astype(int).to_numpy()

            val_trx_auroc, val_trx_auprc   = safe_auc_metrics(val_trx_y,  val_trx_score)
            test_trx_auroc, test_trx_auprc = safe_auc_metrics(test_trx_y, test_trx_score)

            wandb.log({
                "val_trx_auroc": float(val_trx_auroc) if val_trx_auroc == val_trx_auroc else np.nan,
                "val_trx_auprc": float(val_trx_auprc) if val_trx_auprc == val_trx_auprc else np.nan,
                "test_trx_auroc": float(test_trx_auroc) if test_trx_auroc == test_trx_auroc else np.nan,
                "test_trx_auprc": float(test_trx_auprc) if test_trx_auprc == test_trx_auprc else np.nan,
                "val_trx_n": int(len(val_edges)),
                "test_trx_n": int(len(test_edges)),
            })

            for k_pos in [450, 750, 1500, 3000]:
                N_v, p_v, r_v, f_v, found_v = workload_to_find_k_positives(val_trx_y, val_trx_score, k_pos)
                N_t, p_t, r_t, f_t, found_t = workload_to_find_k_positives(test_trx_y, test_trx_score, k_pos)
                wandb.log({
                    f"val_trx_find_{k_pos}_N_required": N_v,
                    f"val_trx_find_{k_pos}_precision": p_v,
                    f"val_trx_find_{k_pos}_recall": r_v,
                    f"val_trx_find_{k_pos}_f1": f_v,
                    f"val_trx_find_{k_pos}_found_tp": found_v,
                    f"test_trx_find_{k_pos}_N_required": N_t,
                    f"test_trx_find_{k_pos}_precision": p_t,
                    f"test_trx_find_{k_pos}_recall": r_t,
                    f"test_trx_find_{k_pos}_f1": f_t,
                    f"test_trx_find_{k_pos}_found_tp": found_t,
                })

            if (DAY_COL in val_edges.columns) and (DAY_COL in test_edges.columns):
                val_tmp = val_edges[[DAY_COL]].copy()
                val_tmp["y"] = val_trx_y
                val_tmp["score"] = val_trx_score
                daily_v, summary_v = per_day_workload_summary(val_tmp, day_col=DAY_COL, label_col="y", score_col="score")
                wandb.log({
                    "val_trx_perday_table": wandb.Table(dataframe=daily_v),
                    "val_trx_perday_summary": wandb.Table(dataframe=summary_v),
                })

                test_tmp = test_edges[[DAY_COL]].copy()
                test_tmp["y"] = test_trx_y
                test_tmp["score"] = test_trx_score
                daily_t, summary_t = per_day_workload_summary(test_tmp, day_col=DAY_COL, label_col="y", score_col="score")
                wandb.log({
                    "test_trx_perday_table": wandb.Table(dataframe=daily_t),
                    "test_trx_perday_summary": wandb.Table(dataframe=summary_t),
                })

            need_cols = {"From Account", "To Account"}
            if need_cols.issubset(set(test_edges.columns)) and (DAY_COL in test_edges.columns):
                days_test = test_edges[DAY_COL].nunique()
                topN = 30 * int(days_test)
                res = topN_distinct_account_metrics(
                    df_meta=test_edges,
                    y_true=test_trx_y,
                    y_score=test_trx_score,
                    topN=topN,
                    account_cols=("From Account", "To Account"),
                    score_agg="max"
                )
                wandb.log({
                    "test_trx_cum_topN": res["topN_trx"],
                    "test_trx_cum_distinct_accounts": res["distinct_accounts"],
                    "test_trx_cum_tp_accounts": res["tp_accounts"],
                    "test_trx_cum_fp_accounts": res["fp_accounts"],
                    "test_trx_cum_precision_accounts": res["precision_accounts"],
                })

        except Exception as e:
            wandb.log({"trx_kpi_error": str(e)})

    # -------------------------
    # 6) node-level extra metrics (val_res/test_res 기반)
    # -------------------------
    for split_name, res in [("val", val_res), ("test", test_res)]:
        yt = res["y_true"]
        yp = res["y_prob"]

        f1_05, p_05, r_05 = f1_at_threshold(yt, yp, 0.5)
        thr_r90 = threshold_for_recall(yt, yp, target_recall=0.9)
        f1_r90, p_r90, r_r90 = f1_at_threshold(yt, yp, thr_r90)

        wandb.log({
            f"{split_name}_thr_0.5_f1": float(f1_05),
            f"{split_name}_thr_0.5_precision": float(p_05),
            f"{split_name}_thr_0.5_recall": float(r_05),
            f"{split_name}_thr_r90": float(thr_r90),
            f"{split_name}_thr_r90_f1": float(f1_r90),
            f"{split_name}_thr_r90_precision": float(p_r90),
            f"{split_name}_thr_r90_recall": float(r_r90),
        })

        yhat = (yp >= 0.5).astype(int)
        rep = classification_report(yt, yhat, output_dict=True, zero_division=0)
        rep_df = pd.DataFrame(rep).T.reset_index().rename(columns={"index": "class"})
        wandb.log({f"{split_name}_cls_report_thr0.5": wandb.Table(dataframe=rep_df)})

    # # =======================================================
    # # ★ GNN Permutation Importance 계산 (수동 지정 대체)
    # # =======================================================
    # print("\n📊 Calculating Permutation Importance for GNN...")

    # 0. feature_names가 없다면 강제로 생성 (오류 방지)
    # 만약 전처리 단계에서 저장한 리스트가 있다면 그것을 쓰세요.

    # def get_permutation_importance_gnn(model, loader, device, target_attr, base_ap, f_names):
    #     import copy
    #     importance_results = []

    #     # 텐서를 직접 복사해서 조작 (GNN 특성 반영)
    #     orig_edge_attr = loader.data.edge_attr.clone()

    #     for i, col in enumerate(f_names):
    #         # 1. 특정 피처 치환 (Shuffle)
    #         perm = torch.randperm(orig_edge_attr.size(0))
    #         shuffled_attr = orig_edge_attr.clone()
    #         shuffled_attr[:, i] = orig_edge_attr[perm, i]

    #         # 2. 로더의 데이터를 잠시 교체
    #         loader.data.edge_attr = shuffled_attr

    #         # 3. 성능 측정
    #         y_perm, p_perm = eval_loader(model, loader, device, target_attr)
    #         p_perm = np.nan_to_num(p_perm, nan=0.0)
    #         _, perm_ap = safe_auc_metrics(y_perm, p_perm)

    #         # 4. 하락폭 계산
    #         drop = base_ap - perm_ap
    #         importance_results.append((col, max(0, drop)))

    #         # 5. 원복
    #         loader.data.edge_attr = orig_edge_attr
    #         print(f"  - [{i+1}/{len(f_names)}] {col}: {drop:.4f}")

    #     return sorted(importance_results, key=lambda x: x[1], reverse=True)

    def get_permutation_importance_gatv2(model, data_mp, edges_meta, device, base_ap, f_names, cfg):
        """
        GATv2 전용 치환 중요도 계산:
        1. 엣지 피처를 섞음
        2. 전체 노드 확률 다시 추론 (infer_all_node_probs)
        3. 거래 점수 재산출 (make_trx_scores_from_node_probs)
        4. AUPRC 하락폭 측정
        """
        import copy
        importance_results = []

        # 원본 엣지 피처 백업
        orig_edge_attr = data_mp.edge_attr.clone()

        # 엣지 기반 KPI를 위한 상수 설정
        EDGE_LABEL_COL = "Is Laundering"

        for i, col in enumerate(f_names):
            # [1] 특정 피처 치환 (Shuffle)
            perm = torch.randperm(orig_edge_attr.size(0))
            shuffled_attr = orig_edge_attr.clone()
            shuffled_attr[:, i] = orig_edge_attr[perm, i]

            # [2] 데이터 임시 교체
            data_mp.edge_attr = shuffled_attr

            # [3] 재평가 (GATv2 전용 로직)
            # 노드 확률 추론
            node_probs = infer_all_node_probs(model, data_mp, device, list(cfg.fanout), int(cfg.batch_size))
            # 거래 점수 산출
            trx_scores = make_trx_scores_from_node_probs(edges_meta, node_probs, agg="max")
            trx_y = edges_meta[EDGE_LABEL_COL].astype(int).to_numpy()

            # 성능 측정
            _, perm_ap = safe_auc_metrics(trx_y, trx_scores)
            perm_ap = 0.0 if np.isnan(perm_ap) else perm_ap

            # [4] 하락폭 계산
            drop = base_ap - perm_ap
            importance_results.append((col, max(0, drop)))

            # [5] 원복
            data_mp.edge_attr = orig_edge_attr
            print(f"  - [{i+1}/{len(f_names)}] {col}: {drop:.4f}")

        return sorted(importance_results, key=lambda x: x[1], reverse=True)

    # # 실행
    # gnn_imp_rows = get_permutation_importance_gnn(model, test_loader, device, test_target_edge_attr, t_ap, feature_names)

    # === [함수 내부 하단에 추가] ===

    # 1. 테스트 결과 데이터 확보 (KPI용 점수)
    # GATv2는 이미 위에서 test_trx_score와 test_trx_y를 계산했습니다.
    y_t = test_trx_y
    p_t = test_trx_score

    # 2. Permutation Importance 계산
    print("\n📊 Calculating Permutation Importance for GATv2...")

    # GATv2NodeClassifier는 forward에서 x, edge_index, edge_attr 등을 따로 받으므로
    # 기존 get_permutation_importance_gnn 함수를 조금 수정해서 호출해야 합니다.
    # (일단은 분석 함수 호출을 위해 필요한 리스트를 생성합니다)
    feature_names = EDGE_ATTR_COLS + [f"cat_{i}" for i in range(len(edge_cat_sizes))]

    # t_ap는 위에서 계산된 test_trx_auprc를 사용합니다.
    t_ap = test_trx_auprc

    # 3. 상세 분석 함수 호출 (test_df 대신 edges_df_meta 사용)
    # edges_df_meta[d.edge_loss_test]가 실제 테스트 거래 데이터프레임입니다.
    df_test_kpi = edges_df_meta[d.edge_loss_test.cpu().numpy()].copy()

    # 패턴 분석 테이블
    t_sum_3000, t_det_3000 = get_pattern_analysis_tables_find_k(df_test_kpi, p_t, 3000)
    wandb.log({
        "GATv2_Analysis/Pattern_Summary_Find3000": t_sum_3000,
        "GATv2_Analysis/Pattern_Detail_Find3000": t_det_3000
    })

    # Find-3000 에러 분석
    log_find_k_error_analysis(
        split_name="test_gat",
        df=df_test_kpi,
        y_true=y_t,
        y_score=p_t,
        k_pos_target=3000,
        top_features=EDGE_ATTR_COLS[:10] # 일단 수치형 피처 위주로 분석
    )

    # # WandB 로깅
    # imp_df = pd.DataFrame(gnn_imp_rows, columns=["feature", "importance"])
    # wandb.log({"gnn_permutation_importance_table": wandb.Table(dataframe=imp_df)})

    # top_features_for_analysis = [x[0] for x in gnn_imp_rows[:10]]

    # # --- (3) 패턴별 검출 성능 (Find-3000 기준) ---
    # t_sum_3000, t_det_3000 = get_pattern_analysis_tables_find_k(test_df, p_t, 3000)
    # wandb.log({
    #     "GNN_Analysis/Pattern_Summary_Find3000": t_sum_3000,
    #     "GNN_Analysis/Pattern_Detail_Find3000": t_det_3000
    # })

    # print(f"[Analysis] Using Top Features: {top_features_for_analysis}")

    # # --- (4) 정탐/오탐/미탐 상세 분석 (Find-3000 기준) ---
    # # 기존에 정의한 log_find_k_error_analysis 함수가 있는 위치를 확인해주세요!
    # log_find_k_error_analysis(
    #     split_name="test_gnn",
    #     df=test_df,
    #     y_true=y_t,
    #     y_score=p_t,
    #     k_pos_target=3000,
    #     top_features=top_features_for_analysis
    # )

    # === 실행부 (train_eval_sweep 함수 맨 끝에 넣으세요) ===
    extract_gat_embeddings(model, d, d.test_mask, device, "gat_node_emb_test.parquet", cfg.hidden_dim)

    if device.type == "cuda":
        torch.cuda.empty_cache()
    run.finish()

# 14-2) Sweep Config 정의 + 실행

In [ ]:
sweep_config_gatv2 = {
    "method": "bayes",
    "metric": {"name": "val_auprc", "goal": "maximize"},
    "parameters": {
        "hidden_dim": {"values": [128]},
        "heads": {"values": [2]},
        "num_layers": {"values": [2]},
        "dropout": {"values": [0.3]},
        "negative_slope": {"values": [0.1]},
        "lr": {"values": [1e-4]},
        "weight_decay": {"values": [1e-4]},
        "max_epochs": {"value": 20},
        "early_patience": {"value": 10},
        "add_self_loops": {"values": [True]},
        "pos_weight_factor": {"values": [1.0]},
        "edge_cat_emb_dim": {"values": [2]},
        "fanout": {"values": [[25, 10]]},
        "batch_size": {"values": [128]},
    }
}

sweep_id = wandb.sweep(sweep_config_gatv2, project="eungyulwon")
wandb.agent(sweep_id, function=train_eval_sweep, count=1)

In [ ]:
# sweep_config = {
#     "method": "bayes",
#     "metric": {"name": "val_auprc", "goal": "maximize"},
#     "parameters": {
#         "hidden_dim": {"values": [128]},
#         "heads": {"values": [2]},
#         "num_layers": {"values": [2]},
#         "dropout": {"values": [0.2]},
#         "negative_slope": {"values": [0.1, 0.2]},
#         "lr": {"values": [3e-4]},
#         "weight_decay": {"values": [1e-4]},
#         "max_epochs": {"value": 20},
#         "early_patience": {"value": 10},
#         "add_self_loops": {"values": [True, False]},
#         "pos_weight_factor": {"values": [1.0, 5.0, 10.0]},
#         "edge_cat_emb_dim": {"values": [4]},
#         "fanout": {"values": [[10, 5], [15, 10], [25, 10]]},
#         "batch_size": {"values": [128, 256]},
#     }
# }

# sweep_id = wandb.sweep(sweep_config, project="eungyulwon")
# wandb.agent(sweep_id, function=train_eval_sweep, count=5)

In [ ]:
# sweep_config = {
#     "method": "bayes",
#     "metric": {"name": "val_auprc", "goal": "maximize"},
#     "parameters": {
#         "hidden_dim": {"values": [128, 256, 512]},
#         "heads": {"values": [2, 4, 8]},
#         "num_layers": {"values": [2, 3, 4]},
#         "dropout": {"values": [0.1, 0.2, 0.4]},
#         "negative_slope": {"values": [0.1, 0.2]},
#         "lr": {"values": [1e-3, 3e-4, 1e-4]},
#         "weight_decay": {"values": [0.0, 1e-4, 1e-3]},
#         "max_epochs": {"value": 50},
#         "early_patience": {"value": 10},
#         "add_self_loops": {"values": [True, False]},
#         "pos_weight_factor": {"values": [0.5, 1.0, 2.0]},
#         "edge_cat_emb_dim": {"values": [4, 8, 16]},
#         "fanout": {"values": [[10, 5], [15, 10], [25, 10]]},
#         "batch_size": {"values": [128, 256]}, # 512, 1024, 2048
#     }
# }

# sweep_id = wandb.sweep(sweep_config, project="eungyulwon")
# wandb.agent(sweep_id, function=train_eval_sweep, count=10)

# 15) (선택) Artifacts "불러오기" 예시 (필요할 때만 실행)
#     - W&B에서 best 모델 버전 지정해서 다시 로컬로 가져올 수 있음

In [ ]:
# run = wandb.init(project="aml-team-experiments", job_type="inference")
# artifact = run.use_artifact("YOUR_ENTITY/aml-team-experiments/aml_xgb_bundle:latest", type="model")
# artifact_dir = artifact.download()
# loaded_transformer = joblib.load(os.path.join(artifact_dir, "transformer.joblib"))
# loaded_model = joblib.load(os.path.join(artifact_dir, "aml_model.joblib"))
# run.finish()

# 이제 loaded_transformer/loaded_model로 동일 파이프라인 재현 가능